<a href="https://colab.research.google.com/github/RochaGerd/Chemistry_with_Python/blob/main/Minicurso_VENEBIOTEC_2026_Parte_02_v_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

|    |    |    |
|:----:|:----:|:----:|
|<img src = "https://github.com/RochaGerd/Chemistry_with_Python/blob/main/figures/Imagem_UFPB.gif?raw=true" width = "150"> | <img src = "https://github.com/RochaGerd/Chemistry_with_Python/blob/main/figures/Imagem_DQ_UFPB.png?raw=true" width = "250"> | <img src = "https://github.com/RochaGerd/Chemistry_with_Python/blob/main/figures/LQQC_2024.png?raw=true" width = "300">|

# **[Quimioinformática e Modelagem Molecular — Perspectivas para a Pesquisa em Saúde](https://www.even3.com.br/v-encontro-nacional-de-biotecnologia-da-renorbio-646346/)**

### **Parte 02 de 3 — Docking e dinâmica molecular**

**Minicurso teórico-prático do V ENEBIOTEC / RENORBIO**
18 de agosto de 2026, bloco das 15:40 às 17:00

**Prof. Gerd Bruno Rocha** — gbr@academico.ufpb.br
Departamento de Química, UFPB · Laboratório de Química Quântica Computacional

*Versão 4.0 — 16, agosto de 2026*

---

<font color='red'>**Este notebook precisa de GPU e de uma cópia sua.**</font>

1. `Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware → GPU (T4)`
2. `Arquivo → Salvar uma cópia no Drive`

Trocar o acelerador depois reinicia a sessão e apaga tudo o que já foi rodado. Faça isso agora, antes da primeira célula.

---

| Onde me encontrar | |
|---|---|
| Grupo de pesquisa | [www.quantum-chem.pro.br](http://www.quantum-chem.pro.br) |
| GitHub | [RochaGerd/Chemistry_with_Python](https://github.com/RochaGerd/Chemistry_with_Python) |
| Lattes | [lattes.cnpq.br/9404945858555096](http://lattes.cnpq.br/9404945858555096) |
| ORCID | 0000-0001-9805-9497 |
| Instagram | [@lqqc.ufpb](https://www.instagram.com/lqqc.ufpb/) |

## **Como este notebook está organizado**

Mesma convenção da Parte 01:

| Cor do título | O que significa |
|---|---|
| <font color='green'>**Verde**</font> | Conteúdo que vamos trabalhar durante o minicurso. Rode essas células. |
| <font color='magenta'>**Magenta (opcional)**</font> | Material de referência ou desafios. Não vamos rodar em sala, mas fica aqui para você consultar depois. |

- **Rode as células em ordem.** Este notebook é mais encadeado que o da Parte 01: quase toda célula depende de um arquivo gerado antes dela.
- **Use o índice lateral** para navegar. As seções opcionais já vêm recolhidas.
- Se aparecer `command not found` ou `NameError`, quase sempre é célula anterior que não rodou.

---

# <font color='green'> **0. Apresentação e roteiro**</font>

## <font color='green'>**0.1 Onde estamos**</font>

Na primeira etapa, a estrutura molecular foi codificada em formato textual (SMILES), convertida em descritores numéricos e submetida à geração de sua conformação tridimensional. A fase anterior encerrou-se com a consolidação das coordenadas atômicas $3\text{D}$ do ligante isolado. A partir deste ponto, vamos considerar a interação da molécula frente ao seu receptor-alvo.

Teremos, então, duas atividades, com lógicas diferentes:

**Atividade 1 — docking molecular.** Atracação de ligantes no sítio ativo de uma proteína e quantificação do quão bem eles cabem. É um cálculo estático: a proteína fica parada, o ligante procura a melhor posição, e uma função de pontuação atribui um número a cada tentativa. Rápido o suficiente para rodar em milhares de compostos.

**Atividade 2 — dinâmica molecular.** O sistema vai evoluir no tempo, a partir da resolução das equações de Newton passo a passo. Os átomos têm energia cinética. É caro computacionalmente, mas responde a perguntas que o docking não consegue responder: a pose se mantém? A proteína muda de forma? Uma cadeia se enovela sozinha? A segunda atividade usa a TRP-cage, uma miniproteína de vinte resíduos que enovela em microssegundos e virou o sistema-modelo padrão para estudar enovelamento. Vamos partir da cadeia esticada e assistir ela se dobrar um pouco.

---

## <font color='green'>**0.2 Roteiro da Parte 02**</font>

```
┌──────────────────────────────────────────────────────────────────────────┐
│  1. PREPARANDO O AMBIENTE                                                │
│     GPU T4 · gnina · Open Babel · RDKit · py3Dmol                        │
└─────────────────────────────────────┬────────────────────────────────────┘
                                      ▼
╔══════════════════════════════════════════════════════════════════════════╗
║  ATIVIDADE 1 — Docking molecular com o gnina                             ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║   2. Desenho baseado  ──►   3. Conceitos e     ──►   4. Docking e        ║
║      em estrutura            sistema (ERK2)            validação         ║
║                                                             │            ║
║        ┌────────────────────────────────────────────────────┘            ║
║        ▼                                                                 ║
║   5. Quanto custa:    ──►   6. Funções de      ──►   7. Quando o         ║
║      exaustividade           pontuação (opc.)          docking fica      ║
║                                                        difícil           ║
║                                                             │            ║
║        ┌────────────────────────────────────────────────────┘            ║
║        ▼                                                                 ║
║   8. Triagem virtual  ──►   9. Função de pontuação própria (opcional)    ║
║      e curvas ROC                                                        ║
║                                                             │            ║
║        ┌────────────────────────────────────────────────────┘            ║
║        ▼                                                                 ║
║  10. RESUMO — o que o docking faz e o que não faz                        ║
╚═════════════════════════════════════╤════════════════════════════════════╝
                                      ▼
╔══════════════════════════════════════════════════════════════════════════╗
║  ATIVIDADE 2 — Dinâmica molecular                                        ║
╠══════════════════════════════════════════════════════════════════════════╣
║  11. Enovelamento da TRP-cage com OpenMM                                 ║
║      OpenMM · ff14SB · GB-Neck2 · MDTraj                                 ║
╚═════════════════════════════════════╤════════════════════════════════════╝
                                      ▼
┌──────────────────────────────────────────────────────────────────────────┐
│  12. ENCERRAMENTO                                                        │
│      resumo  · ponte para a Parte 03 · avaliação · referências           │
└──────────────────────────────────────────────────────────────────────────┘
```

---

**Material original da Atividade 1:** notebook do workshop de gnina apresentado por **David Ryan Koes** (University of Pittsburgh) na série *RSC CICAG Open Source Tools for Chemistry Workshops*. [Vídeo](https://www.youtube.com/watch?v=MG3Srzi5kZ0) · [Software](https://github.com/gnina/gnina)

**Material original da Atividade 2:** adaptado de [TRP Cage OpenMM](https://dev.simonduerr.eu/interactive/mdmc/Ex6/TRP_Cage_OpenMM_colab.html), de Simon Dürr.

---


## <font color='green'>**0.3 Cronograma do minicurso**</font>

| Horário | Bloco | Tempo estimado |
|---|---|:---:|
| 14:00 - 14:40 | **Parte teórica** | 40 min|
| 14:40 – 15:40 | **Parte 01** — Quimioinformática, CADD e QSAR | 60 min |
| 15:40 – 15:50 | Intervalo | 10 min |
| 15:50 – 17:00 | **Parte 02** — Docking e dinâmica molecular | 70 min |
| 17:00 – 17:25 | **Parte 03** — Cálculos quânticos com MOPAC | 25 min |
| 17:25 – 17:30 | Encerramento e avaliação | 5 min |


***Os tempos são estimativas***

---

## <font color='green'>**0.4 Antes de rodar qualquer coisa**</font>

Duas exigências que não são negociáveis.

**A primeira é GPU.** O gnina usa redes neurais convolucionais para pontuar poses, e sem placa gráfica a diferença de tempo é de uma ordem de grandeza. A dinâmica molecular do módulo 11 fica pior ainda. Se você ainda não trocou o acelerador, faça agora: `Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware → GPU (T4)`.

**A segunda é paciência na instalação.** O executável do gnina 1.3.2 tem cerca de 1,4 GB. Ele carrega os pesos das redes neurais embutidos, e é por isso que é tão grande. Numa conexão do Google isso leva de um a três minutos, mas não estranhe a barra de progresso parecendo travada.

## <font color='green'>**0.5 Sobre a autoria, as ferramentas e o uso de IA**</font>

> ***Disclaimer***: **muitos dos códigos aqui apresentados foram retirados de portais, repositórios, livros, apostilas e tutoriais e adaptados, em alguns casos, para melhorar a apresentação do minicurso/disciplina. Em alguns casos, foram feitas correções e atualizações nos códigos para melhor funcionamento nos sistemas operacionais atuais, sempre alinhado com o propósito deste minicurso/disciplina. Também, em algumas partes, foram feitas consultas a *chatbots* especializados em produção de códigos (Gemini, Claude e ChatGPT). Nesses casos, foram feitas testagem e correção dos códigos. Ainda se recorreu a *chatbots* para organização de tabelas, equações e formatações de textos em markdown. Tudo foi checado ao final e atestado para manter o rigor científico, acadêmico e didático. Por todo o conteúdo, o autor desse notebook assume total responsabilidade.**

A declaração acompanha as orientações vigentes do CNPq e do MEC sobre transparência no uso de inteligência artificial generativa em atividades de pesquisa e de ensino: a ferramenta é declarada, o uso é descrito, e a responsabilidade pelo conteúdo permanece integralmente humana.

Registram-se igualmente os programas de código aberto sobre os quais este material se apoia: gnina, AutoDock Vina, Open Babel, RDKit, OpenMM, MDTraj, PDBFixer, BioPython, PeptideBuilder e py3Dmol. Todos estão citados na seção de referências.

## <font color='magenta'>**0.6 Créditos e material de apoio (opcional)**</font>

**Docking e triagem virtual**

- [Documentação do gnina](https://github.com/gnina/gnina) — instalação, opções e exemplos
- [AutoDock Vina](https://vina.scripps.edu/) — o programa de docking mais usado no mundo, base da amostragem do gnina
- [Pharmit](https://pharmit.csb.pitt.edu/) — triagem por farmacóforo em bibliotecas grandes, do mesmo grupo
- [TeachOpenCADD](https://projects.volkamerlab.org/teachopencadd/) — tutoriais de descoberta de fármacos assistida por computador
- [Practical Cheminformatics](https://practicalcheminformatics.blogspot.com/) — análise crítica de métodos, de Pat Walters
- [<font color='red'>**Drug Design Org**</font>](https://www.drugdesign.org/) — É um portal com um extenso conteúdo didático.

**Dinâmica molecular**

- [Documentação do OpenMM](http://docs.openmm.org/) e o [OpenMM Cookbook](https://openmm.github.io/openmm-cookbook/)
- [MDTraj](https://www.mdtraj.org/) — análise de trajetórias
- [Making it rain](https://github.com/pablo-arantes/Making-it-rain) — notebooks de dinâmica molecular em solvente explícito no Colab, de Pablo Arantes
- [Living Journal of Computational Molecular Science](https://livecomsjournal.org/) — boas práticas revisadas por pares, leitura obrigatória antes de publicar simulação

**Bancos de estruturas**

- [RCSB PDB](https://www.rcsb.org/) · [PDBe](https://www.ebi.ac.uk/pdbe/) · [AlphaFold DB](https://alphafold.ebi.ac.uk/)

---

# <font color='green'> **1. Preparando o ambiente**</font>

## <font color='green'>**1.1 Conferindo a GPU**</font>

Antes de instalar 1,4 GB de software, vale conferir se a máquina que o Google entregou tem placa gráfica. Se a célula abaixo disser que não tem, troque o acelerador agora — depois de instalar tudo, trocar reinicia a sessão e você perde o trabalho.

In [ ]:
#@title <font color='green'> 1.1.1 — Conferindo o ambiente e a GPU
# Rode antes de instalar qualquer coisa.
# Se a GPU aparecer como "nenhuma", troque o acelerador AGORA:
# Ambiente de execução -> Alterar o tipo de ambiente de execução -> GPU (T4)

import sys, os, time, platform, subprocess, multiprocessing

print("Data e hora :", time.ctime())
print("Python      :", sys.version.split()[0])
print("Sistema     :", platform.platform())
print("Núcleos CPU :", multiprocessing.cpu_count())

try:
    import google.colab
    print("Ambiente    : Google Colab")
except ImportError:
    print("Ambiente    : local (fora do Colab)")

mem = subprocess.run("free -h | awk '/Mem:/ {print $2}'", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("RAM         :", mem or "não identificada")

disco = subprocess.run("df -h /content 2>/dev/null | awk 'NR==2 {print $4\" livres de \"$2}'",
                       shell=True, capture_output=True, text=True).stdout.strip()
print("Disco       :", disco or "não identificado")

gpu = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
    shell=True, capture_output=True, text=True)

print()
if gpu.returncode == 0 and gpu.stdout.strip():
    print("GPU         :", gpu.stdout.strip())
    print("\nTudo certo. Pode seguir para a instalação.")
else:
    print("GPU         : NENHUMA")
    print("\n" + "!" * 60)
    print("Troque o acelerador ANTES de instalar. Fazer isso depois")
    print("reinicia a sessão e apaga tudo o que já foi rodado.")
    print("Ambiente de execução -> Alterar o tipo de ambiente de execução -> GPU (T4)")
    print("!" * 60)

## <font color='green'>**1.2 Instalando as bibliotecas [3-5 min]**</font>

Open Babel, py3Dmol, RDKit, ProLIF e molview. A instalação do gnina vem depois, porque é a mais demorada.

Uma observação sobre a ordem dos comandos na célula: o pacote `openbabel`, ao ser importado, define as variáveis `BABEL_LIBDIR` e `BABEL_DATADIR`. Sem elas os executáveis não acham seus plugins e falham com `Unable to LoadAllPlugins`. Por isso o import vem antes de mexer no `PATH`.

In [ ]:
#@title <font color="green">**1.2.1 — Instalando bibliotecas [3-5 min]**</font>

import os, sys, warnings

print("Instalando Open Babel...")
os.system("pip install openbabel-wheel")

print("Instalando py3Dmol...")
os.system("pip install py3Dmol")

print("Instalando bibliotecas complementares...")
os.system("pip install rdkit")
os.system("pip install prolif")
os.system("pip install prolif[tutorials]")
os.system("pip install molview")

warnings.filterwarnings("ignore")

print("\nConferindo os imports...")
faltando = []
try:
    import rdkit
    # Vinculamos os DOIS nomes de propósito: algumas células deste notebook
    # usam py3Dmol.view(...) e outras usam p3d.view(...).
    import py3Dmol
    p3d = py3Dmol
    import prolif as plf
    import molview as mv
    print("  [ok] rdkit, py3Dmol, prolif, molview")
except ModuleNotFoundError as e:
    faltando.append(str(e))
    print(f"  [x ] {e}")

if faltando:
    print("\nAlgo falhou. Rode numa célula nova: print(sys.path)")
else:
    print("\nTudo carregado. Siga para a célula 1.3.")

## <font color='green'>**1.3 Conferindo a instalação**</font>

Cartão de visitas do ambiente. Os dois comandos do Open Babel que usaremos o tempo todo são o `obabel`, que converte formatos e gera geometrias 3D, e o `obrms`, que calcula desvio quadrático médio entre estruturas. Se algum deles não responder, o módulo 4 não funciona.

In [ ]:
#@title <font color='green'> 1.3.1 — Open Babel e conferência do ambiente
# A ordem aqui importa. Ao ser importado, o pacote openbabel define as
# variáveis BABEL_LIBDIR e BABEL_DATADIR, sem as quais os executáveis não
# encontram seus plugins e falham com "Unable to LoadAllPlugins".

import os, sys, glob, gzip, warnings
warnings.filterwarnings("ignore")

import openbabel
BIN_OB = os.path.join(os.path.dirname(openbabel.__file__), "bin")
os.environ["PATH"] = BIN_OB + ":" + os.environ["PATH"]

from openbabel import pybel

print("Open Babel :", pybel.ob.OBReleaseVersion())
print("py3Dmol    :", p3d.__version__)
print("executáveis:", BIN_OB)

print("\n--- Os dois comandos que usaremos o tempo todo ---")
!obabel -V
!obrms 2>&1 | head -2

## <font color='green'>**1.4 Baixando o gnina [1-3 min]**</font>

O gnina é distribuído como um executável único, estaticamente ligado, que já carrega dentro de si os pesos das redes neurais treinadas. Daí o tamanho de 1,4 GB. A contrapartida é boa: não existe dependência externa para instalar nem ambiente para configurar, apenas um arquivo para tornar executável.

A versão 1.3.3 é a mais recente (JUN-26). Ela migrou o mecanismo de aprendizado profundo de Caffe para PyTorch, ficou mais rápida e trouxe funções de pontuação destiladas, pensadas para triagem em larga escala. Se você quiser reproduzir exatamente o workshop original do Koes, a versão de lá era a 1.0.1.

De toda forma, você pode querer baixar a versão 1.3.2. Para isso mude a variável `VERSAO` no script a seguir, que está previsto para usar apenas as versões 1.3.2 e 1.3.3, ambas para GPU.



In [ ]:
#@title <font color='green'> 1.4.1 — Baixando o executável do gnina
import os

VERSAO = "1.3.3"  # Opções suportadas: "1.3.2" ou "1.3.3"

# Mapeamento de nomes de arquivo de acordo com a release
NOMES_ARQUIVOS = {
    "1.3.2": "gnina.1.3.2.cuda12.8",
    "1.3.3": "gnina.cuda12.8.static",
}

if VERSAO not in NOMES_ARQUIVOS:
    raise ValueError(
        f"Versão '{VERSAO}' não suportada. Escolha '1.3.2' ou '1.3.3'."
    )

arquivo_origem = NOMES_ARQUIVOS[VERSAO]
url_download = f"https://github.com/gnina/gnina/releases/download/v{VERSAO}/{arquivo_origem}"

if not os.path.exists("gnina"):
    print(f"Baixando GNINA {VERSAO} (CUDA 12.8, aguarde)...")
    get_ipython().system(f"wget -q --show-progress {url_download} -O gnina")
    get_ipython().system("chmod +x gnina")
else:
    print("gnina já está presente nesta sessão.")

tamanho = os.path.getsize("gnina") / 1e9
print(f"\nTamanho do executável: {tamanho:.2f} GB")

In [ ]:
#@title <font color='green'> 1.4.2 — Confirmando a instalação
# Se esta célula imprimir a versão, a instalação está completa.
!./gnina --version

O ponto e a barra antes do nome são obrigatórios. Eles dizem ao sistema para procurar o executável na pasta atual, e não nos diretórios padrão do sistema. Sem isso você recebe `command not found`, mesmo com o arquivo ali na sua frente.

# <font color='green'>**2. Desenho de fármacos baseado em estrutura**</font>

A Parte 01 tratou da situação em que se conhecem ligantes e não o alvo. Aqui a informação se inverte: a estrutura tridimensional do receptor é conhecida e o problema passa a ser desenhar uma molécula complementar a ela.

Esta seção detalha minimamente alguns conceitos sobre aspectos relacionados ao docking molecular.

## <font color='green'>**2.1 A estrutura como hipótese de desenho**</font>

A diferença entre as duas abordagens não se limita ao foco. No desenho baseado em ligante a inferência é estatística, extrapolada de um conjunto rotulado. No desenho baseado em estrutura a inferência é **mecanicista**: a estrutura fornece uma hipótese explícita sobre quais interações existem, onde e por quê.

Isso altera a natureza do que se pode concluir. Um modelo baseado em ligante afirma que um composto se assemelha a ativos conhecidos. Um modelo baseado em estrutura afirma que determinado grupo carbonila aceita uma ligação de hidrogênio de determinada treonina.

A grandeza-alvo é a energia livre de ligação:

$$\Delta G_{\text{lig}} = \Delta H_{\text{lig}} - T\Delta S_{\text{lig}}$$

A estrutura informa diretamente o termo entálpico, por meio de contatos, ligações de hidrogênio e complementaridade eletrostática. O termo entrópico, dominado por dessolvatação e perda de graus de liberdade, permanece o mais difícil de estimar. Essa assimetria explica por que métodos baseados em estrutura acertam a pose com frequência muito maior do que acertam a afinidade.

A vantagem estratégica decisiva está em que a abordagem não fica restrita à química já conhecida do alvo. Um método baseado em ligante busca o que se assemelha ao que já se tem; um método baseado em estrutura pode propor uma plataforma farmacoquímica a ser explorada, desde que ela seja vantajosa com o ambiente do sítio de ligação.

## <font color='green'>**2.2 De onde vem a estrutura**</font>

| Origem | O que fornece | Limitação principal |
|---|---|---|
| Cristalografia de raios X | alta resolução, com ligante quando cocristalizado | conformação do estado cristalino; contatos de empacotamento |
| Crio-microscopia eletrônica | complexos grandes, membranares, múltiplos estados | resolução no sítio nem sempre suficiente para desenho |
| Ressonância magnética nuclear | ensemble em solução, informação dinâmica | limitada a proteínas pequenas |
| Predição por aprendizado profundo | cobertura de praticamente qualquer sequência | conformação única, sítio menos confiável, sem águas nem cofatores |

A crio-microscopia eletrônica atingiu resolução atômica em 2020, o que tornou acessíveis ao desenho racional classes inteiras de alvos antes fora de alcance: receptores acoplados a proteína G em estados de sinalização específicos, canais iônicos e complexos macromoleculares. A diferença prática está em observar o alvo no estado funcional relevante, e não apenas na forma que cristaliza.

Quanto a estruturas preditas, o resumo operacional é que elas ampliaram consideravelmente o conjunto de alvos abordáveis sem substituir o cristal onde ele existe. O rotâmero das cadeias laterais do sítio é a parte menos confiável do modelo, e é justamente ela que define a forma da cavidade do sítio de ligação.

Uma recomendação válida para qualquer origem: estrutura **holo**, com ligante no sítio, é preferível à **apo**. A cavidade de uma proteína apo frequentemente se apresenta colapsada ou parcialmente ocupada por água ordenada, e ancorar contra ela subestima o que o sítio pode acomodar. A estrutura 3ERK que usaremos no módulo 3 é holo, e essa é uma boa escolha.

## <font color='magenta'>**2.3 Encontrar e caracterizar o sítio**</font>

Quando o sítio de ligação não é conhecido previamente, ele precisa ser identificado geometricamente. A abordagem mais difundida utiliza a tesselação de Voronoi e a triangulação de Delaunay sobre o conjunto de coordenadas atômicas para definir **esferas alfa**: esferas tangentes a exatamente quatro átomos e totalmente livres de outros átomos em seu interior.

O raio da esfera alfa fornece uma métrica de profundidade direta: esferas muito pequenas situam-se no interior compacto e inacessível do *core* proteico, enquanto esferas muito grandes representam a superfície exposta ao solvente. Filtrando por uma faixa intermediária de raios e agrupando as esferas remanescentes por proximidade espacial, mapeiam-se as cavidades da proteína — princípio fundamental do algoritmo *fpocket* e de ferramentas correlatas.

Uma vez localizado o sítio, o passo seguinte é avaliar sua **druggabilidade** (*druggability*), isto é, a capacidade de ser modulado eficazmente por uma molécula pequena. Essa análise combina volume, grau de enclausuramento, hidrofobicidade e rigidez estrutural. Um argumento quantitativo direto deriva da Eficiência do Ligante (LE). Como valores excepcionais de LE situam-se em torno de $0{,}5\ \text{kcal/mol}$ por átomo pesado, uma molécula padrão de 25 átomos pesados dificilmente apresentará uma energia livre de ligação mais favorável que:

$$\Delta G_{\text{lig}} \approx -0{,}5 \times 25 = -12{,}5\ \text{kcal/mol}$$

o que estabelece um limite de afinidade na faixa nanomolar. Se a cavidade comporta apenas um ligante pequeno e apresenta caráter predominantemente polar e exposto, a afinidade máxima alcançável já estará severamente restringida antes mesmo da síntese do primeiro composto. Mapear esse teto termodinâmico na etapa de análise estrutural exige baixo custo computacional; ignorá-lo pode custar anos de desenvolvimento inócuo.

A caracterização por **pontos quentes** (*hot spots*) refina essa previsão: via mapeamento por sondas solventes pequenas (como no servidor *FTMap*), identificam-se as sub-regiões que concentram a maior energia de interação. Sítios promissores apresentam poucos pontos quentes, bem definidos e contíguos, permitindo que um ligante de tamanho razoável adicione contribuições entálpicas e entrópicas cooperativas sem dispersar interações no espaço.

A célula abaixo torna esse teto explícito.

## <font color='magenta'>**2.4 Termodinâmica da água no sítio**</font>

Este é o aspecto do desenho baseado em estrutura mais distante da intuição e, quando bem explorado, um dos mais produtivos.

A cavidade não está vazia antes da chegada do ligante: está preenchida por água. Ligar significa **deslocar essa água**, e o custo ou ganho desse deslocamento entra diretamente na energia livre. Uma molécula de água que estabelece três ligações de hidrogênio bem satisfeitas numa região polar é cara de remover. Uma molécula confinada numa cavidade hidrofóbica fechada, incapaz de formar sua rede normal, possui energia livre alta, e sua remoção **rende** afinidade.

Cabe registrar a implicação prática imediata: na seção 3.4 descartaremos todas as águas do arquivo cristalográfico, procedimento usual e adequado na maioria dos alvos. Existem casos, porém, em que uma água estrutural faz ponte entre proteína e ligante, e mantê-la é a diferença entre reproduzir a pose correta e não reproduzir nada.

## <font color='green'>**2.5 As duas abordagens, lado a lado**</font>

| Critério | Design Baseado em Estrutura (SBDD) | Design Baseado em Ligante (LBDD) |
| --- | --- | --- |
| **Informação de entrada** | Estrutura tridimensional do alvo biológico (PDB/Crio-EM) | Conjunto de moléculas ativas conhecidas com dados de afinidade |
| **Fundamento da inferência** | Mecanicista e testável (baseado em físico-química) | Estatístico e indutivo (relacionamento estrutura-atividade) |
| **Inovação química (*Scaffold Hopping*)** | Alta (independe de ligantes conhecidos, limitada pela biblioteca) | Moderada (restrita à vizinhança estrutural dos ativos conhecidos) |
| **Predição da *pose* de ligação** | Direta e validável por cristalografia | Indireta (baseada em superposição ou hipótese farmacofórica) |
| **Custo computacional** | Elevado (exige simulações e *docking*) | Baixo (baseado em alinhamento estrutural e *fingerprints*) |
| **Gargalo principal** | Qualidade da estrutura 3D e exatidão da função de pontuação | Qualidade, diversidade e viés do conjunto de ativos conhecidos |

---

A principal razão para combinar **SBDD** e **LBDD** reside na **ortogonalidade de seus erros**: por operarem sob princípios teóricos totalmente distintos, as duas abordagens raramente falham pelas mesmas razões.

Como detalhado na Seção 3.6 (Parte 01), a eficiência do consenso em triagem virtual não depende apenas da acurácia individual de cada método, mas da independência de suas fontes de erro. Por essa razão, filtrar a biblioteca selecionando os compostos situados na **interseção dos melhores ranqueamentos de ambas as ferramentas** proporciona um taxa de enriquecimento de acertos (*hit rate*) significativamente superior à otimização isolada de qualquer uma delas.

---

# <font color='green'>**3. Docking molecular: os conceitos e o sistema-modelo**</font>

## <font color='green'>**3.1 Amostragem e pontuação**</font>

Programas de docking fazem duas coisas, e é útil separá-las mentalmente porque elas falham de maneiras diferentes.

A **amostragem** gera poses possíveis do ligante dentro do sítio: posição, orientação e conformação interna. É um problema de busca num espaço de muitas dimensões, resolvido por métodos estocásticos.

A **pontuação** atribui a cada pose um número que estima quão boa ela é. É um problema de física aproximada, e é aqui que mora a maior parte do erro.


```
  ┌──────────────┐
  │   RECEPTOR   │──────┐
  │   rec.pdb    │      │      ┌──────────────────┐
  └──────────────┘      ├─────►│  CAIXA DE BUSCA  │
                        │      │ --autobox_ligand │
  ┌──────────────┐      │      └────────┬─────────┘
  │   LIGANTE    │──────┘               │
  │lig.pdb/SMILES│                      ▼
  └──────────────┘         ┌───────────────────────────┐
                           │        AMOSTRAGEM         │
                           │    busca estocástica      │
                           │  gera poses candidatas    │
                           └─────┬───────────────┬─────┘
                                 │               │
                 ┌───────────────┘               └───────────────┐
                 ▼                                               ▼
   ┌──────────────────────────┐                    ┌──────────────────────────┐
   │   PONTUAÇÃO CLÁSSICA     │                    │    PONTUAÇÃO NEURAL      │
   │  soma de termos físicos  │                    │ rede convolucional 3D    │
   │  vina · vinardo · ad4    │                    │ CNNscore · CNNaffinity   │
   └────────────┬─────────────┘                    └────────────┬─────────────┘
                └──────────────────┐   ┌────────────────────────┘
                                   ▼   ▼
                      ┌────────────────────────────┐
                      │      POSES ORDENADAS       │
                      │        docked.sdf          │
                      └─────────────┬──────────────┘
                                    ▼
                      ┌────────────────────────────┐
                      │        VALIDAÇÃO           │
                      │    RMSD contra o cristal   │
                      │     critério:  < 2 Å       │
                      └────────────────────────────┘
```

O AutoDock Vina, provavelmente o programa de docking mais usado no mundo, faz as duas com métodos clássicos: busca estocástica para amostrar, função empírica de termos físicos para pontuar.

O **gnina** mantém a amostragem herdada do Vina e troca a pontuação por um conjunto de redes neurais convolucionais treinadas em estruturas cristalográficas de complexos proteína-ligante. A rede recebe a pose como uma grade tridimensional de densidades atômicas, literalmente enxerga o complexo em 3D, e devolve dois números: a probabilidade de aquela pose estar correta e uma estimativa de afinidade.



## <font color='green'>**3.2 Teoria: a base termodinâmica do docking**</font>

Antes de executar o primeiro cálculo, convém mostrar o que o docking aproxima.

### <font color='green'>**3.2.1 A energia livre de ligação**</font>

Dado um receptor de estrutura tridimensional conhecida e um ligante, o objetivo do *docking* é prever a geometria do complexo e a força da associação. A grandeza termodinâmica fundamental desse processo é a energia livre padrão de ligação:

$$\Delta G^\circ_{\text{lig}} = -RT \ln K_a^\circ = RT \ln K_d^\circ$$

Em projetos de fármacos, a constante de dissociação ($K_d$) varia desde a escala milimolar ($\text{mM}$), para fragmentos moleculares, até a picomolar ($\text{pM}$), para inibidores otimizados. A $298\text{ K}$, a constante $RT \ln 10 \approx 1{,}36\ \text{kcal/mol}$ estabelece que cada ordem de grandeza de variação em $K_d$ exige apenas $1{,}36\ \text{kcal/mol}$ de estabilização termodinâmica. A margem de erro é extremamente estreita: uma desvio de apenas $2{,}72\ \text{kcal/mol}$ na função de pontuação equivale a errar a constante de afinidade por um fator de cem ($10^2$).



### <font color='green'>**3.2.2 A dificuldade combinatória da busca**</font>

Para um ligante tratado como corpo flexível, a dimensionalidade do espaço de configurações é

$$d = \underbrace{3}_{\text{translação}} + \underbrace{3}_{\text{orientação}} + \underbrace{N_{\text{rot}}}_{\text{torções}}$$

A magnitude do problema merece ser quantificada. Numa caixa de 22 Å de aresta com resolução de 0,375 Å existem cerca de $2\times10^5$ posições de translação. Amostrando orientações a cada 10°, obtêm-se cerca de $10^4$ orientações. Três mínimos por ligação rotacionável produzem $3^{N_{\text{rot}}}$ conformações. Para um ligante com oito ligações rotacionáveis, o produto ultrapassa $10^{13}$ configurações.

A avaliação exaustiva está, portanto, fora de questão, e é por isso que a busca é sempre heurística e estocástica. Segue daí uma consequência prática de primeira importância: **executar o mesmo docking duas vezes não produz o mesmo resultado**. A verificação de convergência por réplicas independentes não é refinamento opcional, e é o que trataremos na seção 4.2.

### <font color='magenta'>**3.2.3 Algoritmos de busca**</font>

**Pré-cálculo em grade.** A ideia que viabilizou o docking moderno vem do programa GRID, de Goodford (1985), e foi incorporada ao DOCK por Meng, Shoichet e Kuntz (1992). Como o receptor é mantido rígido, o potencial que ele gera pode ser pré-calculado uma única vez numa grade tridimensional, para cada tipo de átomo do ligante. Durante a busca, a energia de interação de um átomo sai por interpolação trilinear dos oito vértices vizinhos, em lugar de uma soma sobre milhares de átomos do receptor. O ganho é de duas a três ordens de grandeza em velocidade, e é o que torna possível a triagem virtual de milhões de compostos. O preço é a rigidez do receptor: uma vez calculada, a grade não muda.

**Construção incremental.** O ligante é fragmentado; uma âncora rígida é posicionada no sítio e os demais fragmentos são acrescentados progressivamente, com poda das ramificações desfavoráveis. É a estratégia do FlexX (Rarey e colaboradores, 1996). Determinística e rápida, porém sensível à escolha da âncora: se ela for mal posicionada, nenhuma extensão posterior recupera a pose correta.

**Monte Carlo com recozimento simulado.** Perturbações aleatórias aceitas pelo critério de Metropolis,

$$P_{\text{aceite}} = \min\left[1,\; e^{-\Delta E/k_BT}\right]$$

com temperatura fictícia reduzida ao longo da corrida. A aceitação ocasional de movimentos desfavoráveis é o que permite escapar de mínimos locais.

**Algoritmos genéticos.** A configuração é codificada num cromossomo — posição, orientação e ângulos de torção — e uma população evolui por seleção, cruzamento e mutação. É o motor do GOLD (Jones e colaboradores, 1997). O AutoDock 4 emprega um algoritmo genético lamarckiano (Morris e colaboradores, 1998), no qual a minimização local aplicada a um indivíduo é escrita de volta em seu genótipo.

**Busca local iterada.** O AutoDock Vina (Trott e Olson, 2010) alterna perturbação aleatória global com minimização local quase-Newton do tipo BFGS, usando gradientes analíticos do escore. Essa combinação, somada ao paralelismo em múltiplos núcleos, explica o ganho de velocidade e de acurácia de pose em relação ao AutoDock 4. É a busca que o gnina herda e que utilizaremos.

| Programa | Estratégia de busca | Natureza |
|---|---|---|
| DOCK | complementaridade de forma e crescimento incremental | determinística |
| FlexX | construção incremental a partir de âncora | determinística |
| GOLD | algoritmo genético | estocástica |
| AutoDock 4 | algoritmo genético lamarckiano | estocástica |
| AutoDock Vina | busca local iterada com otimização BFGS | estocástica |
| Glide | funil hierárquico de filtros com refinamento por campo de força | híbrida |

O parâmetro `exhaustiveness`, que exploraremos no módulo 5, controla o número de corridas independentes da busca. Aumentá-lo reduz a variância entre execuções, sem corrigir um erro da função de pontuação: uma busca melhor encontra o mínimo da função errada com mais confiabilidade.

## <font color='green'>**3.3 O sistema-modelo: a quinase ERK2**</font>

Vamos trabalhar com a estrutura **3ERK** do Protein Data Bank: a ERK2 (quinase 2 regulada por sinal extracelular) de rato, em complexo com o inibidor SB220025.

A escolha é boa para fins didáticos por três motivos. A ERK2 é uma proteína quinase, classe de alvo que responde por uma fatia enorme dos fármacos oncológicos aprovados nas últimas duas décadas. O sítio de ligação de ATP é uma cavidade bem definida, o que torna a caixa de busca fácil de justificar. E existe uma segunda estrutura da mesma proteína com outro ligante, a 4ERK, que usaremos no módulo 7 para um exercício de cross-docking, aquela situação realista em que você não tem a estrutura co-cristalizada do seu composto.

In [ ]:
#@title <font color='green'> 3.3.1 — Baixando a estrutura 3ERK
# O PDB entrega a estrutura completa: proteína, ligante, águas e íons juntos.
!wget -q http://files.rcsb.org/download/3ERK.pdb
!ls -lh 3ERK.pdb

## <font color='green'>**3.4 Separando receptor e ligante**</font>

Um arquivo PDB cristalográfico vem com as coordenadas do receptor e do ligante, ou seja de um complexo ligante-receptor. As linhas que começam com `ATOM` descrevem a cadeia polipeptídica; as que começam com `HETATM` descrevem o restante: ligante, águas, íons, co-fatores.

Para o docking precisamos dos dois lados separados, o receptor de um lado e o ligante do outro. A separação a seguir é feita com o `grep`, que filtra linhas por padrão de texto.

O `obabel` entra depois de extrair o receptor para adicionar os hidrogênios que a cristalografia de raios X não enxerga e padronizar o arquivo. Sem hidrogênios, os termos de ligação de hidrogênio da função de pontuação ficam sem sentido.

In [ ]:
#@title <font color='green'> 3.4.1 — Separando receptor e ligante
# Receptor: apenas as linhas ATOM (a cadeia proteica), depois padronizado
!grep ATOM 3ERK.pdb > rec.pdb
!obabel rec.pdb -O rec.pdb 2>/dev/null

# Ligante: o SB4 é o código de três letras do inibidor SB220025 neste PDB
!grep 'SB4' 3ERK.pdb > lig.pdb

# 3. Exibe a contagem de átomos nos novos arquivos
print("receptor:", sum(1 for l in open("rec.pdb") if l.startswith(("ATOM", "HETATM"))), "átomos")
print("ligante :", sum(1 for l in open("lig.pdb") if l.startswith(("ATOM", "HETATM"))), "átomos")

Repare no que foi descartado nessa separação: todas as águas. Descartar água é o procedimento mais comum em docking, e na maior parte dos alvos funciona. Existem casos, porém, em que uma água estrutural faz ponte entre proteína e ligante, e mantê-la é a diferença entre reproduzir a pose correta e não reproduzir nada.

Preparação de receptor é onde mora a maior parte dos erros de docking. Na grande maioria dos casos é a a etapa que decide o resultado.

## <font color='green'>**3.5 Visualizando o complexo**</font>

Duas visualizações do mesmo complexo. A primeira usa py3Dmol, que é o visualizador leve que já vinha da Parte 01. A segunda usa o molview, que traz um painel de controle interativo para trocar estilos e cores sem editar código.

Arraste com o mouse para girar, role para dar zoom.

In [ ]:
#@title <font color='green'> 3.5.1 — Visualizando o complexo com py3Dmol
# Visualização do complexo cristalográfico.
# Arraste com o mouse para girar, role para dar zoom.

v = py3Dmol.view(width=700, height=480)
v.addModel(open("rec.pdb").read(), "pdb")
v.setStyle({"cartoon": {}, "stick": {"radius": 0.12}})
v.addModel(open("lig.pdb").read(), "pdb")
v.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.25}})
v.setBackgroundColor("white")
v.zoomTo({"model": 1})
v.show()

In [ ]:
#@title <font color='green'> 3.5.2 — O mesmo complexo com o molview (painel interativo)
import molview as mv

# Visualizador com painel de controle
vv = mv.view(width=600, height=500, panel=True)

vv.addModel(open("3ERK.pdb").read(), name="3ERK")
#vv.addModel(open("lig.pdb").read(), name='lig')

# Exibe
#vv.setColorMode('rainbow', palette='viridis')
vv.setColorMode('secondary', helix_color='#FF6B6B', sheet_color='#4ECDC4', coil_color='#FFE66D')
vv.show()

## <font color='green'>**3.6 Do SMILES à estrutura tridimensional**</font>

Até aqui trabalhamos com a pose que veio do cristal. Num projeto real essa situação é rara: o que existe é o desenho da molécula em 2D, e é preciso gerar coordenadas.

A opção `--gen3D` do Open Babel roda uma busca conformacional e devolve uma geometria tridimensional plausível. É o mesmo problema que resolvemos com o ETKDG do RDKit no exercício 4 da Parte 01, agora por outro caminho.

O SMILES usado é do próprio SB220025, o mesmo ligante do cristal. No módulo 4 vamos dockar essa geometria gerada e comparar com a cristalográfica. É um teste que rende: se o programa reencontra a pose partindo de uma conformação inventada, o protocolo está validado.

In [ ]:
#@title <font color='green'> 3.6.1 — Gerando coordenadas 3D a partir do SMILES

SMILES = "C1CNCCC1n1cnc(c2ccc(cc2)F)c1c1ccnc(n1)N"

# --gen3D: busca conformacional e otimização com campo de força
!obabel -:'{SMILES}' -Ol3.sdf --gen3D 2>/dev/null

v = py3Dmol.view(width=560, height=380)
v.addModel(open("l3.sdf").read(), "sdf")
v.setStyle({"stick": {"colorscheme": "greenCarbon"}})
v.setBackgroundColor("white")
v.zoomTo()
v.show()
print("Geometria tridimensional, pronta para docking.")

Uma observação que costuma passar batida: a conformação que o `--gen3D` entrega é de baixa energia **no vácuo**, e quase certamente não é a conformação bioativa. Ligantes normalmente se ligam numa conformação de 1 a 5 kcal/mol acima do mínimo, porque a proteína paga esse custo em troca de interações favoráveis.

Isso não é problema para o docking, que explora a flexibilidade do ligante durante a busca. Mas é a razão pela qual não adianta caprichar demais na otimização da geometria de entrada.

# <font color='green'> **4. Docking simples e validação**</font>

## <font color='green'>**4.1 O primeiro docking**</font>

Chegamos ao que interessa. A linha abaixo é um docking completo, e vale destrinchar cada pedaço porque ela reaparece no notebook inteiro com pequenas variações.

- `-r rec.pdb` indica o receptor.
- `-l lig.pdb` indica o ligante a ser encaixado.
- `--autobox_ligand lig.pdb` define a caixa de busca automaticamente, envolvendo o ligante informado com uma folga padrão.

O terceiro argumento merece atenção. Estamos usando o **próprio ligante cristalográfico** para dizer ao programa onde procurar. Isso é legítimo aqui, porque o objetivo é validar o método. Mas é um tipo de informação que você não tem diante de um alvo novo, e no módulo 7 veremos o que acontece quando ela é retirada.

In [ ]:
#@title <font color='green'> 4.1.1 — Primeiro docking
# Docking básico. A saída vai para a tela, e não gravamos as poses ainda.
!./gnina -r rec.pdb -l lig.pdb --autobox_ligand lig.pdb

A tabela impressa traz uma linha por pose, ordenada da melhor para a pior. As colunas são:

- **affinity** — estimativa de energia livre de ligação, em kcal/mol, pela função clássica herdada do Vina. Mais negativo é melhor.
- **intramol** — energia interna do ligante naquela conformação.
- **CNNscore** — probabilidade, entre 0 e 1, de aquela pose estar correta, segundo a rede neural. É o número que distingue o gnina de um programa de docking convencional.
- **CNNaffinity** — afinidade estimada pela rede, em unidades de pKd.

Repare que a ordem pela affinity clássica e a ordem pelo CNNscore nem sempre coincidem. Quando divergem, vale olhar as duas poses: é aí que o método de aprendizado profundo está discordando do método físico, e a discordância é informativa.

## <font color='green'>**4.2 Fixando a semente e gravando as poses**</font>

Docking é um método estocástico. A busca parte de configurações aleatórias, e duas execuções da mesma linha de comando devolvem resultados um pouco diferentes. Numa sala com trinta pessoas rodando a mesma célula, isso vira um problema didático: metade vê um número, a outra metade vê outro, e gera um complicador acerca do software.

A opção `--seed 0` fixa o gerador de números aleatórios e torna o resultado reprodutível. Use sempre que quiser comparar execuções. E guarde a ressalva: reprodutível não é sinônimo de correto. Fixar a semente elimina a variação, não o erro.

O `-o docked.sdf.gz` grava as poses num arquivo comprimido.

In [ ]:
#@title <font color='green'> 4.2.1 — Docking reprodutível
!./gnina -r rec.pdb -l lig.pdb --autobox_ligand lig.pdb --seed 0 -o docked.sdf.gz

In [ ]:
#@title <font color='green'> 4.2.2 — Descomprimindo e olhando o resultado
# Descomprimimos porque algumas versões do Open Babel têm dificuldade
# com arquivos SDF comprimidos.
!gunzip -f docked.sdf.gz
!head -20 docked.sdf

# Para ver o arquivo inteiro, descomente a linha abaixo (saída longa):
#!more docked.sdf

## <font color='green'>**4.3 As quatro capacidades do docking e o critério de 2 Å**</font>

Para avaliar a acurácia de um protocolo de *docking* sem ambiguidades, a literatura — consolidada no *benchmark* CASF (Su *et al.*, 2019) — categoriza o desempenho em **quatro capacidades independentes**. Confundi-las é uma das principais fontes de interpretações incorretas na área.

| Capacidade | O que mede | Como se avalia |
| --- | --- | --- |
| **Poder de *docking*** | Capacidade de reproduzir a pose cristalográfica de referência. | Porcentagem de casos em que o RMSD da pose predita é $\le 2\ \text{\AA}$. |
| **Poder de pontuação** | Capacidade de prever a energia livre ou constante de afinidade absoluta ($\Delta G$, $K_d$ ou $K_i$). | Coeficiente de correlação (Pearson ou Spearman) entre a pontuação e os dados experimentais. |
| **Poder de ranqueamento** | Capacidade de ordenar corretamente a potência relativa de ligantes do mesmo alvo. | Correlação de ordem ($\rho$ de Spearman ou $\tau$ de Kendall) em séries congêneres. |
| **Poder de triagem** | Capacidade de discriminar moléculas ativas de inativas (*decoys*) em grandes bibliotecas. | Fator de Enriquecimento (EF) e Área Sob a Curva ROC (AUC-ROC). |


*Esta seção foca exclusivamente no **Poder de docking**; a avaliação do **Poder de triagem** é detalhada no Módulo 8.*

---

### O Cálculo do RMSD

A métrica quantitativa para avaliar o **Poder de *docking*** é o Desvio Quadrático Médio (*Root Mean Square Deviation* — RMSD) entre as posições dos $N$ átomos pesados da pose predita ($\mathbf{r}_i^{\text{predita}}$) e da pose experimental ($\mathbf{r}_i^{\text{experimental}}$):

$$\text{RMSD} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}\left\Vert{}\mathbf{r}_i^{\text{predita}}-\mathbf{r}_i^{\text{experimental}}\right\Vert{}^2}$$

---

### A Limitação do Critério de $2,0\mathring{A}$

O valor limite de $\text{RMSD} \le 2,0\mathring{\mathrm{A}}$ estabeleceu-se como uma convenção prática na comunidade, mas não representa uma fronteira física rigorosa. Esse limiar apresenta um viés importante ligado ao tamanho molecular: $2\ \mathring{\mathrm{A}}$ é um critério bastante amplo para fragmentos pequenos, porém desproporcionalmente rigoroso para ligantes grandes e flexíveis. Por esse motivo, abordagens recentes sugerem a adoção complementar de métricas de desvio normalizadas pelo número de átomos pesados.

---

## <font color='green'>**4.4 Redocking: a validação**</font>

Este é um procedimento muito importante em estratégias de docking molecular.

O `obrms` calcula o desvio quadrático médio entre duas estruturas: a pose que o programa calculou e a pose que o cristalógrafo determinou experimentalmente. A opção `--firstonly` manda usar apenas a primeira molécula do **arquivo de referência**, de modo que todas as poses do arquivo de teste sejam comparadas contra ela. Como o `lig.pdb` tem uma molécula só, aqui a opção não muda nada — ela serve de proteção para quando a referência tem vários modelos, o que acontece com estruturas de RMN.

A convenção da área é que RMSD abaixo de **2 Å** significa que o programa reproduziu a pose experimental. Se o gnina não consegue reproduzir a pose do ligante que veio no próprio cristal, não existe razão nenhuma para acreditar nas poses que ele propõe para compostos novos.

Esse teste se chama redocking, custa trinta segundos e é ignorado com uma frequência que assusta.

Vale baixar o arquivo `.sdf` gerado e olhar por dentro. É texto puro: cada bloco é uma pose, e as propriedades calculadas ficam listadas ao final de cada uma.

In [ ]:
#@title <font color='green'> 4.4.1 — Redocking: RMSD contra o cristal
!obrms lig.pdb docked.sdf --firstonly

In [ ]:
#@title <font color='green'> 4.4.2 — Comparando calculado e experimental
# As poses calculadas sobre a estrutura cristalográfica.
# Cinza: pose experimental.  Verde: poses calculadas, em animação.

v = py3Dmol.view(width=720, height=500)
v.addModel(open("rec.pdb").read(), "pdb")
v.setStyle({"cartoon": {}, "stick": {"radius": 0.08}})

v.addModel(open("lig.pdb").read(), "pdb")
v.setStyle({"model": 1}, {"stick": {"colorscheme": "dimgrayCarbon", "radius": 0.14}})

v.addModelsAsFrames(open("docked.sdf", "rt").read(), "sdf")
v.setStyle({"model": 2}, {"stick": {"colorscheme": "greenCarbon"}})

v.animate({"interval": 1200})
v.setBackgroundColor("white")
v.zoomTo({"model": 1})
v.rotate(90)
v.show()

## <font color='green'>**4.5 Exercício: dockando a conformação gerada**</font>

Em vez de dockar `lig.pdb`, que é a conformação cristalográfica, vamos dockar `l3.sdf`, a conformação que geramos do zero a partir do SMILES. É a situação real de um projeto: você tem o desenho da molécula, não a estrutura do complexo.

A caixa continua definida pelo ligante cristalográfico. Estamos testando a capacidade de reencontrar a pose, não de encontrar o sítio.

In [ ]:
#@title <font color='green'> 4.5.1 — Docking da conformação gerada
!./gnina -r rec.pdb -l l3.sdf --autobox_ligand lig.pdb --seed 0 -o docked_gen.sdf

print("\n--- RMSD da pose calculada contra a cristalográfica ---")
!obrms --firstonly lig.pdb docked_gen.sdf

> **Sobre os avisos `ligand outside box` no log:** eles aparecem durante as etapas aleatórias de exploração do algoritmo e indicam apenas que o ligante saiu momentaneamente dos limites da caixa durante a busca local. É comportamento normal e não compromete o resultado.

Compare este RMSD com o do redocking da seção 4.2. É esperado que este seja um pouco pior: partindo de uma conformação arbitrária, o programa precisa acertar ao mesmo tempo a posição, a orientação e a conformação interna do ligante. Se os dois ficaram abaixo de 2 Å, o protocolo está validado para este alvo.

## <font color='magenta'>**4.6 Desafio 1**</font>

1. Rode a célula 4.1 três vezes seguidas, sem `--seed`. Quanto varia a affinity da melhor pose? E o CNNscore? Essa variação é menor ou maior que a diferença entre duas poses consecutivas da mesma execução?
2. Troque `--seed 0` por `--seed 42` e refaça o redocking. O RMSD muda? Se mudar muito, o que isso diz sobre a confiança que você pode depositar num único valor?
3. Pegue no [PubChem](https://pubchem.ncbi.nlm.nih.gov/) o SMILES de um inibidor de quinase que você conheça, gere a estrutura 3D com `--gen3D` e docke na ERK2. O escore é comparável ao do SB220025? (Cuidado ao interpretar: veja a seção 9.)

---

# <font color='green'> **5. Quanto custa: exaustividade e paralelismo**</font>

## <font color='green'>**5.1 O efeito da exaustividade**</font>

Todo docking é um compromisso entre qualidade do cálculo e tempo de execução. O parâmetro que controla esse compromisso é a **exaustividade**, que define quantas buscas independentes o programa executa antes de escolher a melhor. O padrão do gnina é 8.

As células abaixo medem o efeito. A *magic* `%%time` do Jupyter cronometra a execução, e o redirecionamento para `/dev/null` descarta a saída para que a tela mostre apenas o tempo.

In [ ]:
%%time
!./gnina -r rec.pdb -l lig.pdb --autobox_ligand lig.pdb --seed 0 --exhaustiveness 1 > /dev/null 2>&1

In [ ]:
%%time
!./gnina -r rec.pdb -l lig.pdb --autobox_ligand lig.pdb --seed 0 --exhaustiveness 4 > /dev/null 2>&1

Agora a comparação que mostra de onde vem o desempenho: a mesma exaustividade, restringindo o programa a um único núcleo de processador.

In [ ]:
%%time
!./gnina -r rec.pdb -l lig.pdb --autobox_ligand lig.pdb --seed 0 --exhaustiveness 4 --cpu 1 > /dev/null 2>&1

In [ ]:
#@title <font color='green'> 5.1.1 — Recursos da máquina
# Quantos núcleos a máquina desta sessão tem?
!grep -c ^processor /proc/cpuinfo
!grep "model name" /proc/cpuinfo | head -1

## <font color='green'>**5.2 Duas conclusões práticas**</font>

A primeira é que a exaustividade escala o tempo quase linearmente, e a qualidade não. Aumentar de 8 para 64 multiplica o custo por oito e melhora pouco. Existe um ponto de retorno decrescente e, para a maioria dos alvos, ele fica entre 8 e 16.

A segunda é que a amostragem é paralelizada entre os núcleos disponíveis, e o Colab entrega máquinas com pouquíssimos núcleos. Se um dia você rodar isso num cluster, a diferença é grande, e vale medir antes de dimensionar uma campanha de triagem.

## <font color='magenta'>**5.3 Opções para ir além (opcional)**</font>

**Forçar o uso exclusivo da CPU:** acrescente a flag `--no_gpu`.

```bash
!./gnina -r rec.pdb -l lig.pdb --autobox_ligand lig.pdb --seed 0 --exhaustiveness 4 --cpu 1 --no_gpu
```

**Escolher uma GPU específica** em sistemas com mais de uma: use `--gpu_ids`.

```bash
!./gnina -r rec.pdb -l lig.pdb --autobox_ligand lig.pdb --seed 0 --exhaustiveness 4 --gpu_ids 0
```

# <font color='green'>**6. Funções de pontuação**</font>

## <font color='green'>**6.1 A função clássica**</font>

Antes de olhar as redes neurais, vale entender o que elas substituem.

A opção `--score_only` pede ao gnina que pontue a pose fornecida sem sair procurando outras. É útil para avaliar uma pose que você já tem, vinda de um cristal, de outro programa ou de uma simulação de dinâmica molecular. Com `--verbosity=2` ele detalha as contribuições.

In [ ]:
#@title <font color='green'> 6.1.1 — Pontuando a pose cristalográfica</font>
!./gnina --score_only -r rec.pdb -l lig.pdb --verbosity=2

In [ ]:
#@title <font color='green'>6.1.2 — Funções de pontuação disponíveis
# Quais funções de pontuação estão disponíveis?
!./gnina --help | grep scoring | head -4

As três opções principais são `vina`, `vinardo` e `ad4_scoring`. A **vinardo** é uma reparametrização da função do Vina publicada em 2016, geralmente melhor para triagem virtual, e é a que usaremos no módulo 8. A `ad4_scoring` reproduz a função do AutoDock 4.

## <font color='green'>**6.2 Abrindo a caixa-preta**</font>

A função de pontuação do Vina não é um número mágico: é uma soma ponderada de termos, cada um representando um tipo de interação física. O comando abaixo lista todos os termos que o gnina sabe calcular.

In [ ]:
#@title <font color='green'> 6.2.1 — Termos disponíveis
!./gnina --print_terms

A célula seguinte grava um arquivo com todos esses termos, cada um com peso 1,0. Isso não é uma função de pontuação sensata — é um instrumento de medida. Ao pontuar com esse arquivo, o gnina reporta o valor individual de cada termo em vez de uma soma ponderada.

Vamos usar isso duas vezes: aqui, para enxergar a decomposição de uma interação; e em seguida, para alimentar um modelo de aprendizado de máquina com esses termos como variáveis.

O texto à direita de cada linha é comentário, ignorado pelo programa. Vale ler: é a documentação mais direta que existe sobre o que cada termo significa.

In [ ]:
#@title <font color='green'> 6.2.2 — Arquivo de pontuação customizada
open("everything.txt", "wt").write('''
1.0  ad4_solvation(d-sigma=3.6,_s/q=0.01097,_c=8)  desolvation, s/q is charge dependence
1.0  ad4_solvation(d-sigma=3.6,_s/q=0.0,_c=8)
1.0  electrostatic(i=1,_^=100,_c=8)	i is the exponent of the distance, see everything.h for details
1.0  electrostatic(i=2,_^=100,_c=8)
1.0  gauss(o=0,_w=0.5,_c=8)		o is offset, w is width of gaussian
1.0  gauss(o=3,_w=2,_c=8)
1.0  repulsion(o=0,_c=8)	o is offset of squared distance repulsion
1.0  hydrophobic(g=0.5,_b=1.5,_c=8)		g is a good distance, b the bad distance
1.0  non_hydrophobic(g=0.5,_b=1.5,_c=8)	value is linearly interpolated between g and b
1.0  vdw(i=4,_j=8,_s=0,_^=100,_c=8)	i and j are LJ exponents
1.0  vdw(i=6,_j=12,_s=1,_^=100,_c=8) s is the smoothing, ^ is the cap
1.0  non_dir_h_bond(g=-0.7,_b=0,_c=8)	good and bad
1.0  non_dir_anti_h_bond_quadratic(o=0.4,_c=8) like repulsion, but for hbond, don't use
1.0  non_dir_h_bond_lj(o=-0.7,_^=100,_c=8)	LJ 10-12 potential, capped at ^
1.0 acceptor_acceptor_quadratic(o=0,_c=8)	quadratic potential between hydrogen bond acceptors
1.0 donor_donor_quadratic(o=0,_c=8)	quadratic potential between hydroben bond donors
1.0  num_tors_div	div constant terms are not linearly independent
1.0  num_heavy_atoms_div
1.0  num_heavy_atoms	these terms are just added
1.0  num_tors_add
1.0  num_tors_sqr
1.0  num_tors_sqrt
1.0  num_hydrophobic_atoms
1.0  ligand_length
''')

print("everything.txt gravado com", len(open("everything.txt").readlines()), "linhas")

In [ ]:
#@title <font color='green'> 6.2.3 — Decompondo a interação em termos
!./gnina -r rec.pdb -l lig.pdb --score_only --custom_scoring everything.txt

Olhe os valores dos termos `gauss` e `hydrophobic`, que costumam dominar. Interações de docking são, em larga medida, complementaridade de forma e contato hidrofóbico. Isso explica por que funções de pontuação acertam razoavelmente bem a geometria da pose e erram bastante a afinidade: forma é fácil de medir, enquanto energia livre de ligação depende de dessolvatação e entropia, que estes termos aproximam de maneira grosseira.

## <font color='green'>**6.3 Pontuação por rede neural**</font>

A ideia é substituir a soma de termos físicos por uma rede neural convolucional que recebe o complexo proteína-ligante como uma grade tridimensional. Cada tipo de átomo vira um canal da grade, do mesmo modo que vermelho, verde e azul são canais de uma imagem. A rede foi treinada no conjunto CrossDocked, aprendendo a distinguir poses corretas de poses plausíveis mas erradas.

O que o gnina entrega não é uma rede só, e sim um conjunto delas. A discordância entre os membros do conjunto vira a `CNNvariance`, que funciona como medida de incerteza. É informação que uma função de pontuação clássica simplesmente não fornece, e vale prestar atenção nela: variância alta significa que o modelo está inseguro, mesmo quando o CNNscore parece bom.

In [ ]:
#@title <font color='green'> 6.3.1 — Modos de pontuação por CNN
# As opções de configuração das redes
!./gnina --help | grep "cnn arg" -A 12

Vale entender os três modos principais de `--cnn_scoring`:

- `none` — desliga a rede e usa apenas a função clássica. É o gnina se comportando como Vina.
- `rescore` — amostra com a função clássica e reordena as poses finais com a rede. É o padrão, e o melhor compromisso entre custo e ganho.
- `refinement` — usa a rede também durante a otimização das poses. Mais caro e, em geral, um pouco melhor.

O modo `rescore` é o padrão justamente porque a amostragem clássica é boa o suficiente para gerar poses candidatas. O que ela faz mal é escolher entre elas, e é aí que a rede entra.

In [ ]:
#@title <font color='green'> 6.3.2 — Pontuação da pose cristalográfica pela rede
!./gnina --score_only -r rec.pdb -l lig.pdb | grep CNN

Um CNNscore próximo de 1 significa que a rede considera a pose quase certamente correta. Como esta é a pose experimental, é o que esperamos. Se não fosse, teríamos motivo para desconfiar do modelo, não do cristalógrafo.

## <font color='magenta'>**6.4 Quanto a GPU importa**</font>

As duas células abaixo rodam exatamente o mesmo docking. A diferença é que a segunda esconde a GPU do programa, forçando o cálculo da rede na CPU.

A variável `CUDA_VISIBLE_DEVICES` vazia é um truque padrão: ela diz ao software que nenhum dispositivo CUDA está disponível, sem precisar desinstalar nada.

In [ ]:
%%time
!./gnina -r rec.pdb -l lig.pdb --autobox_ligand lig.pdb --seed 0 > /dev/null 2>&1

In [ ]:
%%time
!CUDA_VISIBLE_DEVICES= ./gnina -r rec.pdb -l lig.pdb --autobox_ligand lig.pdb --seed 0 > /dev/null 2>&1

A diferença costuma ficar entre cinco e vinte vezes, dependendo da máquina que o Colab entregou. É por isso que ativamos o ambiente com GPU logo no começo.

Vale registrar o que isso significa em escala. Numa triagem virtual de um milhão de compostos, um fator de dez é a diferença entre uma semana e dois meses de máquina. Decisões de infraestrutura em descoberta de fármacos assistida por computador são, com frequência, decisões sobre qual pergunta você consegue fazer.

## <font color='magenta'>**6.5 Desafio 2**</font>

1. Rode o docking do módulo 4 com `--cnn_scoring none` e compare o RMSD com o do padrão (`rescore`). A rede neural ajudou neste alvo?
2. Troque `--scoring vina` por `--scoring vinardo` na pontuação da pose cristalográfica. Os valores são comparáveis entre si? Faz sentido comparar escores de funções diferentes?
3. Na saída da célula 6.3, identifique os três termos com maior valor absoluto. Eles correspondem ao tipo de interação que você esperaria olhando a estrutura do complexo em 3D?

# <font color='green'>**7. Quando o docking fica difícil**</font>

Até agora demos ao programa quase todas as respostas: o sítio certo, o receptor cristalizado com o próprio ligante, a conformação experimental. Cada uma dessas informações vai sair agora, uma de cada vez.

Vale ter em mente a escada de dificuldade. Os números de acerto publicados na literatura caem sensivelmente a cada degrau, e a maior parte dos projetos reais vive nos degraus de baixo.

```
  MAIS FÁCIL
      │
      │   ┌──────────────────────────────────────────────────┐
      │   │ 1. REDOCKING                          módulo 3   │
      │   │    mesmo receptor, mesmo ligante, sítio conhecido│
      │   └──────────────────────────────────────────────────┘
      │        │
      │        ▼
      │       ┌──────────────────────────────────────────────────┐
      │       │ 2. LIGANTE GERADO DO ZERO             módulo 3   │
      │       │    conformação vem do SMILES, sítio conhecido    │
      │       └──────────────────────────────────────────────────┘
      │            │
      │            ▼
      │           ┌──────────────────────────────────────────────────┐
      │           │ 3. CROSS-DOCKING RÍGIDO               módulo 7   │
      │           │    receptor cristalizado com outro ligante       │
      │           └──────────────────────────────────────────────────┘
      │                │
      │                ▼
      │               ┌──────────────────────────────────────────────────┐
      │               │ 4. CROSS-DOCKING FLEXÍVEL             módulo 7   │
      │               │    cadeias laterais do sítio liberadas           │
      │               └──────────────────────────────────────────────────┘
      │                    │
      │                    ▼
      │                   ┌──────────────────────────────────────────────────┐
      │                   │ 5. DOCKING CEGO                       módulo 7   │
      │                   │    caixa na proteína inteira, sítio desconhecido │
      ▼                   └──────────────────────────────────────────────────┘
  MAIS DIFÍCIL

  A cada degrau você retira uma informação do problema, e a taxa de acerto cai.
```

---

## <font color='green'>**7.1 Docking cego, na proteína inteira**</font>

Retiramos a informação do sítio. A mudança é de uma palavra: em vez de `--autobox_ligand lig.pdb`, escrevemos `--autobox_ligand rec.pdb`, e a caixa passa a envolver a proteína inteira. É a situação de quem tem uma estrutura nova e não sabe onde o composto se liga.

Os efeitos práticos são:

1. o cálculo fica bem mais lento, porque o espaço de busca cresce muito.
2. e a taxa de acerto cai: no artigo do GNINA 1.0, o acerto em docking de proteína inteira é substancialmente menor do que com o sítio definido.

Encontrar o sítio é um problema à parte, e não é o docking que o resolve bem.

In [ ]:
#@title <font color='green'> 7.1.1 — Docking na proteína inteira
!./gnina -r rec.pdb -l lig.pdb --autobox_ligand rec.pdb -o wdocking.sdf.gz --seed 0

In [ ]:
#@title <font color='green'> 7.1.2 — Onde as poses foram parar?
import gzip

v = py3Dmol.view(width=720, height=460)
v.addModel(open("rec.pdb").read(), "pdb")
v.setStyle({"cartoon": {}, "stick": {"radius": 0.08}})

v.addModel(open("lig.pdb").read(), "pdb")
v.setStyle({"model": 1}, {"stick": {"colorscheme": "dimgrayCarbon", "radius": 0.14}})

v.addModelsAsFrames(gzip.open("wdocking.sdf.gz", "rt").read(), "sdf")
v.setStyle({"model": 2}, {"stick": {"colorscheme": "greenCarbon"}})

v.animate({"interval": 1200})
v.setBackgroundColor("white")
v.zoomTo()
v.rotate(90)
v.show()

In [ ]:
#@title <font color='green'> 7.1.3 — RMSD do docking cego
# O teste objetivo: alguma das poses reproduz a posição experimental?
!gunzip -kf wdocking.sdf.gz
!obrms --firstonly lig.pdb wdocking.sdf

Compare este RMSD com o do módulo 4. Se a melhor pose foi parar longe do sítio real, a lição está dada: definir o sítio de ligação é metade do trabalho, e informação estrutural prévia vale mais do que poder computacional.

Quando não se tem essa informação, o caminho não é aumentar a caixa e torcer. É detectar a cavidade com ferramentas próprias para isso, como o [fpocket](https://github.com/Discngine/fpocket), ou usar homologia com proteínas da mesma família.

## <font color='green'>**7.2 Cross-docking: o receptor errado**</font>

Todo o docking que fizemos até aqui tratou a proteína como rígida. Essa é a aproximação central do método e também a sua maior fragilidade: proteínas se movem, e o sítio de ligação frequentemente muda de forma para acomodar cada ligante.

Vamos medir o tamanho desse problema. Baixamos a estrutura **4ERK**, que é a mesma quinase ERK2 com um ligante diferente, o olomoucine. Depois tentamos encaixar o ligante do 3ERK dentro do receptor do 4ERK.

É a situação realista de qualquer projeto: você tem a estrutura de um complexo e quer prever como um composto novo se liga. As cadeias laterais do sítio estão na conformação errada, adaptadas ao outro ligante.

In [ ]:
#@title <font color='green'> 7.2.1 — Preparando a segunda estrutura (4ERK)
!wget -q http://files.rcsb.org/download/4ERK.pdb

!grep ATOM 4ERK.pdb > rec2.pdb
!obabel rec2.pdb -Orec2.pdb 2>/dev/null

# OLO é o código do olomoucine, o ligante desta estrutura
!grep OLO 4ERK.pdb > lig2.pdb

print("receptor 4ERK:", sum(1 for l in open("rec2.pdb") if l.startswith(("ATOM","HETATM"))), "átomos")
print("ligante OLO  :", sum(1 for l in open("lig2.pdb") if l.startswith(("ATOM","HETATM"))), "átomos")

In [ ]:
#@title <font color='green'> 7.2.2 — Cross-docking rígido
# Cross-docking com receptor rígido: ligante do 3ERK no receptor do 4ERK
!./gnina -r rec2.pdb -l lig.pdb --autobox_ligand lig2.pdb --seed 0 -o 3erk_to_4erk.sdf

print("\n--- RMSD contra a pose experimental do 3ERK ---")
!obrms --firstonly lig.pdb 3erk_to_4erk.sdf

Este RMSD costuma ser bem pior que o do redocking, e a razão é a rigidez. As cadeias laterais do sítio no 4ERK estão posicionadas para o olomoucine e não abrem espaço para o nosso ligante.

## <font color='magenta'>**7.3 Liberando as cadeias laterais**</font>

O gnina permite tratar parte do receptor como flexível, o que aproxima o cálculo do modelo de encaixe induzido. A opção `--flexdist 4` combinada com `--flexdist_ligand lig2_h.pdb` diz: torne flexíveis todos os resíduos cujas cadeias laterais estejam a menos de 4 Å do ligante de referência. O `--out_flex` grava essas cadeias na conformação encontrada.

Duas advertências. O custo sobe bastante (**mais de 15 minutos !!**), porque cada torção adicional multiplica o espaço de busca. E flexibilidade demais atrapalha: com muitos graus de liberdade, a busca encontra poses que encaixam bem e não têm significado físico.

In [ ]:
#@title <font color='magenta'> 7.3.1 — Docking flexível por distância
!./gnina -r rec2.pdb -l lig.pdb --autobox_ligand lig2.pdb --seed 0 \
    -o flexdocked.sdf --flexdist 4 --flexdist_ligand lig2.pdb --out_flex flexout.pdb

print("\n--- RMSD com cadeias laterais flexíveis ---")
!obrms --firstonly lig.pdb flexdocked.sdf

**O que cada opção faz nessa linha de comando**

| Opção | Efeito |
|---|---|
| `-r rec2.pdb` | receptor, na parte que permanece rígida |
| `-l lig.pdb` | ligante a ser encaixado |
| `--autobox_ligand lig2.pdb` | caixa de busca ao redor da molécula de referência |
| `--seed 0` | semente fixa, para o resultado ser reprodutível |
| `-o flexdocked.sdf` | arquivo com as poses calculadas |
| `--flexdist 4` | libera as cadeias laterais dos resíduos a menos de 4 Å |
| `--flexdist_ligand lig2.pdb` | molécula usada como referência para essa distância |
| `--out_flex flexout.pdb` | arquivo com as conformações finais das cadeias liberadas |

O `obrms --firstonly lig.pdb flexdocked.sdf` compara cada pose gerada contra a conformação cristalográfica de referência e devolve o RMSD em ångströms.

Compare o resultado com o da seção 7.2. Se melhorou, a rigidez das cadeias laterais era mesmo o gargalo. Se não melhorou, a diferença entre as duas estruturas envolve movimento da cadeia principal, que o docking flexível não trata — e nesse caso o caminho é dinâmica molecular ou docking contra um conjunto de conformações.

## <font color='magenta'>**7.4 Escolhendo os resíduos à mão (opcional)**</font>

Em vez de deixar o programa decidir por distância, você pode nomear os resíduos. A sintaxe `A:52,A:103` significa resíduos 52 e 103 da cadeia A.

É a opção que se usa quando existe conhecimento prévio: um resíduo que a literatura descreve como móvel, ou um que a mutagênese apontou como crítico. A exaustividade foi aumentada para 16 aqui porque o espaço de busca é maior.

In [ ]:
#@title <font color='magenta'> 7.4.1 — Docking flexível com resíduos nomeados
!./gnina -r rec2.pdb -l lig.pdb --autobox_ligand lig2.pdb --seed 0 \
    -o flexdocked2.sdf --exhaustiveness 16 \
    --flexres A:52,A:103 --out_flex flexout2.pdb

print("\n--- RMSD com resíduos escolhidos manualmente ---")
!obrms --firstonly lig.pdb flexdocked2.sdf

## <font color='magenta'>**7.5 A escada de dificuldade, em números**</font>

A célula abaixo reúne todos os RMSDs calculados até aqui num gráfico só. É o resumo visual do módulo e, na minha experiência, a figura que mais fica na cabeça de quem está começando.

Ela lê os arquivos já gerados nas seções anteriores, então rode as seções anteriores antes. O que faltar aparece como ausente, sem quebrar o gráfico.

In [ ]:
#@title <font color='green'> 7.5.1 — A escada de dificuldade, em números
# Reúne os RMSDs de todas as situações testadas até aqui.
# Lê os arquivos já gerados; o que faltar aparece como ausente.

import os, re, subprocess
import matplotlib.pyplot as plt
import numpy as np

def rmsd_top(referencia, poses):
    """RMSD da pose de melhor ranque contra a referência. None se falhar."""
    if not (os.path.exists(referencia) and os.path.exists(poses)):
        return None
    r = subprocess.run(f"obrms --firstonly {referencia} {poses}",
                       shell=True, capture_output=True, text=True)
    valores = []
    for linha in r.stdout.splitlines():
        numeros = re.findall(r"[-+]?\d*\.\d+|\d+", linha)
        if numeros:
            valores.append(float(numeros[-1]))   # o RMSD é sempre o último número
    return valores[0] if valores else None

CASOS = [
    ("1. Redocking\n(caso mais fácil)",        "lig.pdb", "docked.sdf"),
    ("2. Ligante gerado\ndo SMILES",           "lig.pdb", "docked_gen.sdf"),
    ("3. Cross-docking\nrígido (4ERK)",        "lig.pdb", "3erk_to_4erk.sdf"),
    ("4. Cross-docking\nflexível",             "lig.pdb", "flexdocked.sdf"),
    ("5. Docking cego\n(proteína inteira)",    "lig.pdb", "wdocking.sdf"),
]

nomes, valores = [], []
print(f"{'situação':34s} {'RMSD (Å)':>10s}")
print("-" * 46)
for rotulo, ref, teste in CASOS:
    v = rmsd_top(ref, teste)
    limpo = rotulo.replace("\n", " ")
    print(f"{limpo:34s} {v:10.2f}" if v is not None
          else f"{limpo:34s} {'ausente':>10s}")
    if v is not None:
        nomes.append(rotulo)
        valores.append(v)

if valores:
    cores = ["#1a9850" if v < 2 else "#d73027" for v in valores]

    fig, ax = plt.subplots(figsize=(9.5, 4.6))
    barras = ax.bar(nomes, valores, color=cores, edgecolor="k", linewidth=0.5, width=0.6)
    ax.axhline(2.0, color="crimson", ls="--", lw=1.6)
    ax.text(len(valores) - 0.4, 2.12, "critério de 2 Å", color="crimson",
            fontsize=10, ha="right")

    for b, v in zip(barras, valores):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.12, f"{v:.2f}",
                ha="center", fontsize=10, fontweight="bold")

    ax.set_ylabel("RMSD contra a pose cristalográfica (Å)")
    ax.set_title("Quanto mais informação você tira, pior o docking fica")
    ax.set_ylim(0, max(max(valores) * 1.25, 3))
    ax.tick_params(axis="x", labelsize=9)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

    print("\nVerde: abaixo de 2 Å, o protocolo reproduziu a pose experimental.")
    print("Vermelho: acima de 2 Å, não reproduziu.")
else:
    print("\nNenhum arquivo encontrado. Rode as seções específicas antes.")

# <font color='green'>**8. Triagem virtual**</font>

Mudamos de pergunta. Até agora perguntávamos *onde* uma molécula se liga. Agora perguntamos *quais* moléculas se ligam, que é a pergunta de quem tem uma biblioteca de compostos e precisa decidir o que testar na bancada.

O alvo muda para a **4PPS** e usamos um conjunto preparado pelo grupo do Koes, com moléculas ativas e inativas conhecidas misturadas. Como sabemos a resposta, dá para medir objetivamente se a função de pontuação separa umas das outras.

## <font color='green'>**8.1 Preparando o alvo e o conjunto de compostos**</font>

In [ ]:
#@title <font color='green'> 8.1.1 — Preparando o alvo 4PPS
!wget -q http://files.rcsb.org/download/4PPS.pdb
!grep ^ATOM 4PPS.pdb > errec.pdb

print("receptor 4PPS:", sum(1 for l in open("errec.pdb") if l.startswith("ATOM")), "átomos")

In [ ]:
#@title <font color='green'> 8.1.2 — Baixando o conjunto de compostos
!wget -q --show-progress http://bits.csb.pitt.edu/files/workshop_minimized_results.sdf.gz

import os
if os.path.exists("workshop_minimized_results.sdf.gz"):
    tam = os.path.getsize("workshop_minimized_results.sdf.gz") / 1e6
    print(f"\nArquivo baixado: {tam:.1f} MB")
else:
    print("\nDOWNLOAD FALHOU. Verifique a conexão ou use uma cópia local.")

## <font color='green'>**8.2 Reavaliando as poses**</font>

Os compostos já vêm com poses geradas. Não vamos redockar tudo, seria caro e desnecessário. Usamos `--minimize`, que ajusta localmente cada pose e a pontua, sem refazer a busca global.

A função escolhida é a **vinardo**. Mesmo assim, o gnina calcula os escores da rede neural em paralelo, e teremos os dois para comparar.

In [ ]:
#@title <font color='green'> 8.2.1 — Minimização e pontuação
!./gnina -r errec.pdb -l workshop_minimized_results.sdf.gz \
    --minimize -o gnina_scored.sdf.gz --scoring vinardo

In [ ]:
#@title <font color='green'> 8.2.2 — Lendo os escores
from openbabel import pybel
import pandas as pd

# Cada molécula do SDF carrega os escores como propriedades de texto.
registros = []
for mol in pybel.readfile("sdf", "gnina_scored.sdf.gz"):
    registros.append({
        "titulo":      mol.title,
        "CNNscore":    float(mol.data["CNNscore"]),
        "CNNaffinity": float(mol.data["CNNaffinity"]),
        "Vinardo":     float(mol.data["minimizedAffinity"]),
    })

escores = pd.DataFrame(registros)

# O rótulo verdadeiro está no próprio nome do composto.
escores["ativo"] = escores.titulo.str.contains("active")

print(f"{len(escores)} compostos | {escores.ativo.sum()} ativos | "
      f"{(~escores.ativo).sum()} inativos")
escores.head(10)

## <font color='green'>**8.3 Medindo a capacidade de discriminação (Curva ROC e AUC)**</font>

A Curva ROC (*Receiver Operating Characteristic*) quantifica a eficiência de um protocolo de triagem virtual ao mensurar a taxa de verdadeiros positivos (compostos ativos recuperados) em função da taxa de falsos positivos (compostos inativos ou *decoys* selecionados) ao longo de todo o ranking de pontuação.

A Área Sob a Curva (**AUC-ROC**) sintetiza essa capacidade discriminatória em uma única métrica escalar:

* **$\text{AUC} = 0{,}5$:** Desempenho nulo, equivalente a um ordenamento aleatório (*chute ao acaso*).
* **$\text{AUC} = 1{,}0$:** Discriminação perfeita, na qual todos os compostos ativos antecedem os inativos no topo do ranking.

---

### Estratégias Comparadas e Pontuação por Consenso

Para avaliar o impacto da função de pontuação na eficiência da seleção, comparam-se três abordagens de ordenamento:

1. **Pontuação Empírica Isolada (*Vinardo*):** Baseada em termos físico-químicos parametrizados.
2. **Aprendizado Profundo Isolado (*CNN Affinity*):** Baseada na previsão de afinidade da rede neural convolucional do GNINA.
3. **Consenso por Soma de Ranks (*Rank-by-Rank*):** Combina as posições obtidas pelas duas ferramentas individuais para gerar um ranking unificado.

A eficácia do consenso fundamenta-se na **ortogonalidade dos erros**: por operarem sob princípios teóricos e matemáticos distintos, as falhas preditivas de funções empíricas e de redes neurais tendem a não coincidir. A interseção dos melhores resultados reduz a taxa de falsos positivos e confere maior robustez ao protocolo do que a aplicação de qualquer método isolado.

---

### Inversão de Sinal para o Cálculo da ROC

Em convenções termodinâmicas, valores de energia livre de ligação mais baixos (mais negativos) indicam interações mais favoráveis e maior afinidade. Contudo, bibliotecas de análise estatística e aprendizado de máquina (como a *Scikit-Learn*) assumem por padrão que **valores numericamente maiores indicam maior probabilidade de pertencer à classe positiva**.

Por esse motivo, aplica-se a inversão de sinal aos escores termodinâmicos (`-escores.Vinardo`), garantindo que os compostos com melhor perfil de ligação ocupem os primeiros postos do ranking no cálculo da curva.

---

In [ ]:
#@title <font color='green'> 8.3.1 — Curvas ROC
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

plt.figure(figsize=(5.5, 5.5), dpi=110)
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, linewidth=1, label="aleatório")

fpr, tpr, _ = roc_curve(escores.ativo, -escores.Vinardo)
plt.plot(fpr, tpr, lw=2, label="Vinardo (AUC = %.2f)" % auc(fpr, tpr))

fpr, tpr, _ = roc_curve(escores.ativo, escores.CNNaffinity)
plt.plot(fpr, tpr, lw=2, label="CNNaffinity (AUC = %.2f)" % auc(fpr, tpr))

consenso = escores.CNNaffinity.rank() + (-escores.Vinardo).rank()
fpr, tpr, _ = roc_curve(escores.ativo, consenso)
plt.plot(fpr, tpr, lw=2, label="Consenso (AUC = %.2f)" % auc(fpr, tpr))

plt.xlabel("taxa de falsos positivos")
plt.ylabel("taxa de verdadeiros positivos")
plt.title("Triagem virtual em 4PPS")
plt.legend(loc="lower right", fontsize=9)
plt.gca().set_aspect("equal")
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Antes de comemorar qualquer AUC alta, uma ressalva que vale para toda a literatura de triagem virtual: conjuntos de teste com ativos e inativos conhecidos costumam ser mais fáceis que a realidade. Os inativos são frequentemente moléculas escolhidas por serem diferentes dos ativos, e um modelo pode separar as duas classes reconhecendo essa diferença trivial em vez de aprender algo sobre ligação.

O que a AUC mede aqui é a separação neste conjunto. Extrapolar para "este método vai funcionar na minha biblioteca" é um salto que exige validação prospectiva.

Isso conversa diretamente com o domínio de aplicabilidade que discutimos no exercício 3 da Parte 01. É o mesmo problema, com outra roupa.

# <font color='magenta'>**9. Construindo sua própria função de pontuação (opcional)**</font>

Neste exercício, em vez de usar uma função de pontuação pronta, extraímos os termos individuais e treinamos um modelo próprio em cima deles.

O arquivo `everything.txt` que foi gerado anteriormente faz o gnina reportar o valor de cada termo separadamente. Esses valores viram variáveis de entrada, o rótulo de ativo ou inativo vira a variável de saída, e uma regressão logística aprende os pesos.

É, de uma forma super simplificada, exatamente como funções de pontuação empíricas são construídas na literatura.

Depende dos seções 6 e 8 terem rodado, porque usa alguns arquivos.

## <font color='magenta'>**9.1 Extraindo os termos (opcional)**</font>

In [ ]:
#@title <font color='magenta'> 9.1.1 — Extraindo os termos
!./gnina -r errec.pdb -l workshop_minimized_results.sdf.gz \
    --score_only --custom_scoring everything.txt > scores.txt 2>&1

!head -20 scores.txt

In [ ]:
#@title <font color='magenta'> 9.1.2 — Montando a tabela de variáveis
import subprocess, io, re

# As linhas que interessam começam com ##. O sed remove o prefixo.
saida = subprocess.check_output("grep '##' scores.txt | sed 's/## //'", shell=True)

# Uso sep=r'\s+' em vez de delim_whitespace, que foi removido no pandas 3.
termos = pd.read_csv(io.BytesIO(saida), sep=r"\s+")

# Os escores da rede aparecem em linhas separadas; recuperamos por expressão regular.
termos[["CNNscore", "CNNaffinity", "CNNvariance"]] = re.findall(
    r"CNNscore: (\S+)\s*CNNaffinity: (\S+)\s*CNNvariance: (\S+)",
    open("scores.txt").read())

termos["ativo"] = termos.Name.str.contains("active")

print(f"{len(termos)} compostos, {termos.shape[1]-2} variáveis")
termos.head()

In [ ]:
#@title <font color='magenta'> 9.1.3 — Separando entrada e saída
X = termos.drop(["Name", "ativo"], axis=1).astype(float)
y = termos.ativo

print("Variáveis de entrada:")
for i, coluna in enumerate(X.columns, 1):
    print(f"  {i:2d}. {coluna}")

## <font color='magenta'>**9.2 Treinando e validando (opcional)**</font>

Um detalhe importante que não pode ser ignorado: avaliar o modelo nos mesmos dados em que ele foi treinado devolve um número sem muita validade estatística. Por isso usamos validação cruzada, em que o modelo é treinado numa parte dos dados e avaliado na parte que ficou de fora, repetidamente.

A função `cross_val_predict` devolve, para cada composto, a previsão feita por um modelo que não o viu durante o treino. É com essas previsões que a curva ROC abaixo é construída.

In [ ]:
#@title <font color='magenta'> 9.2.1 — Modelo próprio com validação cruzada
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

modelo = LogisticRegression(solver="liblinear")
previsto = cross_val_predict(modelo, X, y, method="predict_proba", cv=5)

fpr, tpr, _ = roc_curve(y, previsto[:, 1])

fig, ax = plt.subplots(figsize=(5.2, 5.2), dpi=110)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, lw=1)
ax.plot(fpr, tpr, lw=2, color="crimson",
        label="modelo próprio, validação cruzada (AUC = %.2f)" % auc(fpr, tpr))
ax.set_xlabel("taxa de falsos positivos")
ax.set_ylabel("taxa de verdadeiros positivos")
ax.legend(loc="lower right", fontsize=8)
ax.set_aspect("equal")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## <font color='magenta'>**9.3 O que o modelo aprendeu (opcional)**</font>

In [ ]:
#@title <font color='magenta'> 9.3.1 — O que o modelo aprendeu
# Quais termos o modelo considerou mais informativos?
# É importante padronizar os coeficientes para que a comparação faça sentido.
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

fluxo = make_pipeline(StandardScaler(), LogisticRegression(solver="liblinear"))
fluxo.fit(X, y)

pesos = pd.Series(fluxo[-1].coef_[0], index=X.columns)
pesos = pesos.reindex(pesos.abs().sort_values(ascending=False).index)

print("Termos mais influentes (coeficientes padronizados):\n")
print(pesos.head(10).round(3).to_string())

Compare a AUC deste modelo com a das funções prontas do módulo 8. Se ficou melhor, foi porque ele se ajustou a este alvo específico, o que é ao mesmo tempo a vantagem e o perigo de uma função de pontuação treinada sob medida. Ela funciona bem onde foi treinada e pode não generalizar para outro sistema.

Olhe também os coeficientes. Se termos de contagem, como `num_heavy_atoms`, aparecerem no topo, o modelo pode estar aprendendo principalmente que moléculas maiores pontuam melhor. Isso é um viés conhecido de docking, não um sinal de reconhecimento molecular.

# <font color='green'>**10. Resumo do que foi visto sobre Docking Molecular**</font>

Exploramos bastante a execução com o gnina: instalação, docking simples, validação por redocking, custo computacional, funções de pontuação clássicas e neurais, docking cego, flexibilidade do receptor, triagem virtual e construção de um modelo próprio.

Um resumo honesto do que essas ferramentas entregam.

**Docking acerta razoavelmente bem a pose e mal a afinidade.** A correlação típica entre escore de docking e dado experimental fica entre 0,4 e 0,6 de $R^2$. Isso serve para priorizar o que testar, e não como estimativa de constante de inibição. Quem reporta escore de docking com duas casas decimais como se fosse $\Delta G$ está contando uma história que o método não sustenta.

**A pontuação por rede neural melhora a escolha da pose** de forma consistente e mensurável. Não resolve o problema da afinidade, e é bom desconfiar de qualquer texto que sugira o contrário.

**O redocking é barato e obrigatório.** Se o programa não reproduz a pose cristalográfica do próprio ligante do alvo, os resultados para compostos novos não têm base.

E a limitação que a próxima módulo ataca: **docking é uma foto**. Uma pose com bom escore pode se desfazer nos primeiros nanossegundos de dinâmica. Quem decide isso é o tempo de residência.

Uma dinâmica molecular ligante-recepotos apresenta diversas preparações técnicas e não será abordada nesse notebook.

---

# <font color='green'>**11. Dinâmica molecular: o enovelamento da TRP-cage**

Mudamos de método e de pergunta.

Docking congela a proteína e procura a melhor posição do ligante. Dinâmica molecular faz o oposto: integra as equações de Newton para cada átomo, passo a passo, e deixa o sistema inteiro se mover. As forças vêm do campo de força que discutimos na seção 3.7 da Parte 01 — as mesmas molas, torções e termos de Lennard-Jones, agora derivados para dar aceleração.

O sistema escolhido é a **TRP-cage**, uma miniproteína de vinte resíduos (sequência `NLYIQWLKDGGPSSGRPPPS`) desenhada em 2002 e que serve em muitos testes de algoritmos para estudos de enovelamento. Ela é pequena, enovela rápido, e tem estrutura experimental de RMN disponível (PDB **1L2Y**).

O experimento: partimos da cadeia **totalmente esticada** e deixamos a dinâmica trabalhar. Se der certo, vamos assistir uma proteína se enovelar sozinha, mesmo que em estágio muito inicial.

```
 ┌─────────────────┐   ┌─────────────────┐   ┌──────────────────┐
 │    SEQUÊNCIA    │   │ CADEIA ESTENDIDA│   │  CAMPO DE FORÇA  │
 │ NLYIQWLKDGGPSSG │──►│  PeptideBuilder │──►│ ff14SB + GB-Neck2│
 │      RPPPS      │   │    + PDBFixer   │   │solvente implícito│
 └─────────────────┘   └─────────────────┘   └────────┬─────────┘
                                                      ▼
 ┌─────────────────┐   ┌─────────────────┐   ┌─────────────────┐
 │    TRAJETÓRIA   │   │  DINÂMICA NVT   │   │   MINIMIZAÇÃO   │
 │  trajetoria.dcd │◄──│ 300 K, Langevin │◄──│   DE ENERGIA    │
 │                 │   │  passo de 2 fs  │   │                 │
 └────────┬────────┘   └─────────────────┘   └─────────────────┘
          ▼
 ┌──────────────────────────────────────────────┐
 │                   ANÁLISE                    │      ┌──────────────────┐
 │  RMSD · raio de giro · diedros φ e ψ         │◄─────│ ESTRUTURA NATIVA │
 │  fração de contatos nativos (Q)              │ ref. │  RMN, PDB 1L2Y   │
 └──────────────────────────────────────────────┘      └──────────────────┘
```

Duas escolhas técnicas que barateiam o cálculo o suficiente para caber numa aula. Usamos **solvente implícito** (modelo GB-Neck2), que trata a água como um meio contínuo em vez de moléculas explícitas — isso acelera muito o enovelamento e reduz drasticamente o número de átomos. E usamos o **OpenMM**, que roda na GPU do Colab sem depender de binários externos.

## <font color='green'>**11.1 Instalando o OpenMM [3-5 min]**</font>

OpenMM para a dinâmica, MDTraj para a análise da trajetória, PDBFixer para completar átomos faltantes, BioPython para manipular o PDB e PeptideBuilder para construir a cadeia estendida a partir da sequência.

In [ ]:
#@title <font color="green">**11.1.1 — Instalando o OpenMM e ferramentas de análise [3-5 min]**</font>

import os, sys, warnings

print("Instalando bibliotecas de dinâmica molecular...")
os.system("pip install openmm")
os.system("pip install mdtraj")
os.system("pip install pdbfixer")
os.system("pip install biopython")
os.system("pip install netcdf4")
os.system("pip install PeptideBuilder")

# Caminhos que o kernel do Colab às vezes ignora
caminhos_alvo = [
    "/usr/local/lib/python3.12/site-packages",
    "/usr/local/lib/python3.12/dist-packages",
    "/usr/local/lib/python3.11/site-packages",
    "/usr/local/lib/python3.11/dist-packages",
    "/usr/local/lib/site-packages",
]

print("Ajustando caminhos do sistema...")
for caminho in caminhos_alvo:
    if os.path.exists(caminho) and caminho not in sys.path:
        sys.path.append(caminho)
        print(f"  mapeado: {caminho}")

warnings.filterwarnings("ignore")

print("\nConferindo os imports...")
try:
    import openmm
    # Novamente os dois nomes: as células de análise usam ora md.load(...),
    # ora mdtraj.rmsd(...). Sem esta linha, metade delas dava NameError.
    import mdtraj
    import mdtraj as md
    import pdbfixer
    import Bio.PDB
    import MDAnalysis as mda
    import PeptideBuilder
    print(f"  [ok] OpenMM   {openmm.__version__}")
    print(f"  [ok] MDTraj   {mdtraj.__version__}")
    print(f"  [ok] BioPython {Bio.__version__}")
    print("  [ok] pdbfixer, MDAnalysis, PeptideBuilder")
    print("\nTudo carregado. Siga para a célula 11.2.")
except ModuleNotFoundError as e:
    print(f"  [x ] {e}")
    print("\nRode numa célula nova: print(sys.path)")

## <font color='green'>**11.2 Importações e plataforma de cálculo**</font>

O OpenMM escolhe entre várias plataformas de execução. A função abaixo tenta CUDA (GPU), depois OpenCL, depois CPU, e avisa qual conseguiu. Se cair em CPU, a dinâmica funciona, só que devagar — nesse caso reduza `n_steps` na seção 11.6.

In [ ]:
#@title <font color="green"> 11.2.1 — Importações e plataforma de cálculo
# --- Importações e seleção automática da plataforma de cálculo ---
import warnings, sys
warnings.filterwarnings('ignore')

from openmm.app import *
from openmm import *
from openmm.unit import *

from pdbfixer import PDBFixer
import numpy as np
import matplotlib.pyplot as plt

def escolher_plataforma():
    # tenta usar a mais rápida disponível: CUDA (GPU) > OpenCL > CPU
    for nome in ('CUDA', 'OpenCL', 'CPU'):
        try:
            plat = Platform.getPlatformByName(nome)
            props = {'Precision': 'mixed'} if nome in ('CUDA', 'OpenCL') else {}
            return plat, props, nome
        except Exception:
            continue
    return Platform.getPlatformByName('Reference'), {}, 'Reference'

plataforma, plat_props, plat_nome = escolher_plataforma()
#print("OpenMM versão:", openmm.__version__)
print("Plataforma selecionada:", plat_nome)
if plat_nome in ('CPU', 'Reference'):
    print(">> Sem GPU detectada: a dinâmica funciona, mas devagar. "
          "Considere reduzir 'n_steps' na célula da dinâmica.")

## <font color='green'>**11.3 Montando o sistema**</font>

Precisamos de duas estruturas:

1. A estrutura **nativa** (enovelada), obtida experimentalmente por RMN. Ela serve de **referência** para medir o quanto a simulação se aproxima do enovelamento correto. Nunca entra no cálculo, apenas na análise.
2. A estrutura **estendida** (esticada), que é o **ponto de partida** do experimento.

O arquivo 1L2Y é um conjunto de vários modelos de RMN. Usamos o primeiro como referência.

In [ ]:
#@title <font color="green"> 11.3.1 — Estrutura nativa de referência (1L2Y)
# Estrutura experimental (RMN) ENOVELADA da TRP-cage, baixada do Protein Data Bank.
!wget -q http://files.rcsb.org/download/1L2Y.pdb

# 1L2Y é um conjunto de vários modelos de RMN; usamos o primeiro como referência.
parser = Bio.PDB.PDBParser(QUIET=True)
estrutura_rmn = parser.get_structure('1L2Y', '1L2Y.pdb')
primeiro_modelo = next(estrutura_rmn.get_models())
# 'escritor' em vez de 'io' para não sombrear o módulo io do Python
escritor = Bio.PDB.PDBIO()
escritor.set_structure(primeiro_modelo)
escritor.save('nativa.pdb')
print("Estrutura nativa de referência salva em 'nativa.pdb'")

In [ ]:
#@title <font color="green"> 11.3.2 — Função auxiliar de visualização
# Função auxiliar de visualização estática (substitui as funções com NGLView).
def mostrar_estrutura(arquivo_pdb, titulo="", w=430, h=330, destacar_trp=True):
    if titulo: print(titulo)
    v = p3d.view(width=w, height=h)
    v.addModel(open(arquivo_pdb).read(), 'pdb')
    v.setStyle({'cartoon': {'color': 'spectrum'}})
    if destacar_trp:   # destaca o triptofano 6 -> o "núcleo" da gaiola (cage)
        v.addStyle({'resi': '6'}, {'stick': {'colorscheme': 'orangeCarbon'}})
    v.zoomTo()
    return v.show()

mostrar_estrutura('nativa.pdb',
    "Estrutura NATIVA (enovelada) da TRP-cage — 1L2Y  (Trp6 em laranja)")

Aqui usamos a biblioteca PeptideBuilder, que monta o peptídeo direto da sequência de uma letra em conformação totalmente estendida, sem sair do Python. O PDBFixer completa os átomos pesados que faltam, como o OXT do C-terminal.

In [ ]:
#@title <font color="green"> 11.3.3 — Construindo a cadeia estendida
# --- Construindo a cadeia ESTENDIDA a partir da sequência ---
# Aqui usamos o PeptideBuilder, que constrói o peptídeo diretamente da
# sequência de 1 letra, em conformação totalmente estendida.
import PeptideBuilder, Bio.PDB
sequencia = "NLYIQWLKDGGPSSGRPPPS"   # TRP-cage (TC5b)

estrutura_estendida = PeptideBuilder.make_extended_structure(sequencia)
escritor = Bio.PDB.PDBIO()
escritor.set_structure(estrutura_estendida)
escritor.save('trp_estendida_heavy.pdb')

# PDBFixer completa átomos pesados que faltam (ex.: OXT do C-terminal)
fixer = PDBFixer(filename='trp_estendida_heavy.pdb')
fixer.findMissingResidues(); fixer.findMissingAtoms(); fixer.addMissingAtoms()
PDBFile.writeFile(fixer.topology, fixer.positions, open('trp_estendida_fix.pdb', 'w'))
print("Cadeia estendida construída a partir da sequência.")

## <font color='green'>**11.4 Campo de força e solvente implícito**</font>

Duas escolhas definem os detalhes da simulação.

O **campo de força** é o ff14SB para a proteína, um dos parametrizados da família AMBER, combinado com o **GB-Neck2** para o solvente implícito.

O **integrador** é o de Langevin, que faz o papel de termostato: mantém a temperatura em 300 K acoplando o sistema a um banho térmico com atrito e ruído. O passo de integração é de 2 fs, possível porque as ligações com hidrogênio estão restritas (`constraints=HBonds`).

Sistema pequeno em solvente implícito dispensa raio de corte para as interações não ligadas, daí o `NoCutoff`.

In [ ]:
#@title <font color="green"> 11.4.1 — Campo de força, solvente implícito e sistema
# --- Campo de força, solvente implícito e criação do sistema ---
# ff14SB (proteína) + GB-Neck2 (solvente implícito).

campo_de_forca = ForceField('amber14-all.xml', 'implicit/gbn2.xml')

# Adiciona hidrogênios de forma consistente com o campo de força
pdb_estendida = PDBFile('trp_estendida_fix.pdb')
modeller = Modeller(pdb_estendida.topology, pdb_estendida.positions)
modeller.addHydrogens(campo_de_forca)
PDBFile.writeFile(modeller.topology, modeller.positions, open('topologia.pdb', 'w'))
print("Átomos totais (com H):", modeller.topology.getNumAtoms())

# Sistema: sem cutoff (sistema pequeno em solvente implícito) e restrições nas
# ligações com H (permite passo de integração de 2 fs).
system = campo_de_forca.createSystem(modeller.topology,
                                     nonbondedMethod=NoCutoff,
                                     constraints=HBonds,
                                     hydrogenMass=1.5*amu)

# Termostato de Langevin a 300 K
integrator = LangevinMiddleIntegrator(300*kelvin, 1.0/picosecond, 0.002*picoseconds)

try:
    simulation = Simulation(modeller.topology, system, integrator, plataforma, plat_props)
except Exception as e:
    print("Plataforma", plat_nome, "indisponível em execução; usando CPU.\n", e)
    plataforma = Platform.getPlatformByName('CPU'); plat_props = {}
    simulation = Simulation(modeller.topology, system, integrator, plataforma, plat_props)

simulation.context.setPositions(modeller.positions)
print("Sistema pronto na plataforma:", plataforma.getName())

In [ ]:
#@title <font color="green"> 11.4.2 — Ponto de partida: a cadeia esticada
# Ponto de partida do experimento: a cadeia totalmente esticada.
mostrar_estrutura('topologia.pdb',
    "Ponto de PARTIDA: cadeia totalmente ESTENDIDA (antes de enovelar)")

## <font color='green'>**11.5 Minimização de energia**</font>

Antes da dinâmica, relaxamos a estrutura estendida para remover contatos ruins — átomos muito próximos que gerariam forças enormes e fariam a integração explodir no primeiro passo.

Compare as duas energias impressas. A queda costuma ser de várias ordens de grandeza, e vem quase toda de resolver sobreposições atômicas.

In [ ]:
#@title <font color="green"> 11.5.1 — Minimização de energia
E0 = simulation.context.getState(getEnergy=True).getPotentialEnergy()
simulation.minimizeEnergy(maxIterations=5000)
E1 = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print("Energia potencial antes :", E0)
print("Energia potencial depois:", E1)

pos_min = simulation.context.getState(getPositions=True).getPositions()
PDBFile.writeFile(modeller.topology, pos_min, open('minimizada.pdb', 'w'))
mostrar_estrutura('minimizada.pdb', "Estrutura minimizada (ainda estendida)")

## <font color='green'>**11.6 A dinâmica (o enovelamento acontece aqui)**</font>

Dinâmica a temperatura constante, 300 K, termostato de Langevin, solvente implícito, passo de 2 fs.

Dois parâmetros para ajustar conforme o relógio:

- **`n_steps`** — número de passos de integração. O padrão de 2.500.000 passos × 2 fs dá **5000 ps (5 ns)** de simulação. Numa T4 isso leva algo entre 5 e 10 minutos.
- **`n_save`** — de quantos em quantos passos um frame é gravado na trajetória.

> **Com o tempo curto ou sem GPU:** reduza `n_steps` para 100.000 (200 ps). Você ainda vê o colapso da cadeia, que é a parte mais visual.
> **Querendo um enovelamento mais completo:** aumente para 5.000.000 ou mais e rode em segundo plano. Enovelamento é estocástico, e mais tempo ajuda.

Os *reporters* do OpenMM gravam a trajetória em DCD e imprimem energia, temperatura e velocidade de simulação enquanto o cálculo roda.

In [ ]:
#@title <font color="green"> 11.6.1 — A dinâmica molecular NVT
# --- Parâmetros da dinâmica ---
n_steps = 2500000  #@param {type:"integer"}
n_save  = 2000     #@param {type:"integer"}
# n_steps x 2 fs = duração da simulação.
#   1000000 -> 2000 ps (2 ns), de 5 a 15 min numa T4
#    100000 ->  200 ps, cerca de 1 min (suficiente para ver o colapso)
# n_save: grava 1 frame a cada n_save passos

# Velocidades iniciais sorteadas a 300 K
simulation.context.setVelocitiesToTemperature(300*kelvin)

# Reporters: trajetória (DCD) + acompanhamento (energia, temperatura, velocidade)
simulation.reporters = []
simulation.reporters.append(DCDReporter('trajetoria.dcd', n_save))
simulation.reporters.append(
    StateDataReporter(sys.stdout, n_save*20, step=True, time=True,
                      potentialEnergy=True, temperature=True,
                      progress=True, remainingTime=True, speed=True,
                      totalSteps=n_steps, separator='  |  '))

print("Rodando %d passos (%.0f ps)...\n" % (n_steps, n_steps*0.002))
simulation.step(n_steps)
print("\nDinâmica concluída! Trajetória salva em 'trajetoria.dcd'.")

## <font color='green'>**11.7 Analisando a trajetória**</font>

Carregamos a trajetória com o MDTraj e a exploramos de cinco formas complementares. Cada uma responde a uma pergunta diferente sobre o mesmo conjunto de dados.

In [ ]:
#@title <font color="green"> 11.7.1 — Carregando a trajetória
traj = md.load('trajetoria.dcd', top='topologia.pdb')
nativa = md.load('nativa.pdb')
print("Frames na trajetória:", traj.n_frames, "| Átomos:", traj.n_atoms)

# Eixo de tempo (ps) usado nos gráficos abaixo
tempo_ps = np.arange(traj.n_frames) * n_save * 0.002

### <font color='green'>**11.7.2 Animação da trajetória**</font>

Geramos um PDB multimodelo e o py3Dmol anima o enovelamento. O triptofano 6 aparece em laranja: é um resíduo importante e que dá nome à TRP-cage e o que fica enterrado quando a estrutura se fecha.

In [ ]:
#@title <font color="green"> 11.7.2.1 — Animação do enovelamento
# Alinha os frames e reduz para ~120 frames (animação leve no navegador)
ca = traj.topology.select('name CA')
traj_alin = traj[:]
traj_alin.superpose(traj_alin, 0, atom_indices=ca)

passo = max(1, traj_alin.n_frames // 120)
traj_alin[::passo].save_pdb('traj_anim.pdb')

view = p3d.view(width=650, height=480)
view.addModelsAsFrames(open('traj_anim.pdb').read())
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.addStyle({'resi': '6'}, {'stick': {'colorscheme': 'orangeCarbon'}})  # Trp6
view.zoomTo()
view.animate({'loop': 'forward', 'interval': 80})
view.show()

### <font color='green'>**11.7.3 RMSD em relação à estrutura nativa**</font>

Mede o quanto a estrutura simulada se afasta ou se aproxima da forma experimental. Queda no RMSD significa aproximação do estado nativo.

In [ ]:
#@title <font color="green"> 11.7.3.1 — RMSD contra a estrutura nativa
ca_traj = traj.topology.select('name CA')
ca_nat  = nativa.topology.select('name CA')
traj.superpose(nativa, atom_indices=ca_traj, ref_atom_indices=ca_nat)
rmsd_nativa = mdtraj.rmsd(traj, nativa, atom_indices=ca_traj,
                      ref_atom_indices=ca_nat) * 10.0     # nm -> Å

plt.figure(figsize=(8, 5))
plt.plot(tempo_ps, rmsd_nativa, lw=1.3)
plt.title('Aproximação ao estado nativo (RMSD do Cα vs. 1L2Y)')
plt.xlabel('Tempo (ps)'); plt.ylabel('RMSD (Å)')
plt.tight_layout(); plt.show()
print("RMSD inicial: %.2f Å  ->  RMSD final: %.2f Å" % (rmsd_nativa[0], rmsd_nativa[-1]))

### <font color='green'>**11.7.4 Raio de giro**</font>

Mede o **colapso** da cadeia. Valor alto indica cadeia esticada; a queda indica compactação, o chamado colapso hidrofóbico, que costuma ser a primeira etapa do enovelamento e acontece bem antes de a estrutura secundária se formar.

In [ ]:
#@title <font color="green"> 11.7.4.1 — Raio de giro
rg = mdtraj.compute_rg(traj) * 10.0     # nm -> Å
plt.figure(figsize=(8, 5))
plt.plot(tempo_ps, rg, color='green', lw=1.3)
plt.title('Colapso da cadeia (raio de giro ao longo do tempo)')
plt.xlabel('Tempo (ps)'); plt.ylabel('Raio de giro (Å)')
plt.tight_layout(); plt.show()
print("Rg inicial: %.2f Å  ->  Rg final: %.2f Å" % (rg[0], rg[-1]))

### <font color='green'>**11.7.5 Ângulos diedros φ e ψ**</font>

Os mesmos observáveis que alimentam um gráfico de Ramachandran. Aqui mostramos a evolução dos seis primeiros pares ao longo do tempo, o que deixa visível o momento em que um resíduo encontra sua conformação e para de girar.

In [ ]:
#@title <font color="green"> 11.7.5.1 — Ângulos diedros φ e ψ
_, phi = mdtraj.compute_phi(traj)      # radianos
_, psi = mdtraj.compute_psi(traj)
phi = np.rad2deg(phi); psi = np.rad2deg(psi)

fig, axs = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
for k, ax in enumerate(axs.flat):
    ax.plot(tempo_ps, phi[:, k], '.', ms=2, color='tab:blue', label='φ')
    ax.plot(tempo_ps, psi[:, k], '.', ms=2, color='tab:red',  label='ψ')
    ax.set_title('Ângulo #%d' % (k + 1)); ax.set_ylim(-180, 180)
    if k == 0: ax.legend(fontsize=8)
for ax in axs[-1]:  ax.set_xlabel('Tempo (ps)')
for ax in axs[:, 0]: ax.set_ylabel('Ângulo (°)')
fig.suptitle('Ângulos diedrais φ e ψ do backbone ao longo da trajetória')
plt.tight_layout(); plt.show()

### <font color='green'>**11.7.6 Fração de contatos nativos (Q)**</font>

Provavelmente o melhor indicador único de enovelamento. Definimos os contatos nativos a partir da estrutura de referência — pares de resíduos com átomos pesados a menos de 4,5 Å, separados por pelo menos 3 resíduos na sequência — e medimos quantos desses contatos existem em cada frame.

$Q \approx 0$ significa cadeia estendida. $Q \to 1$ significa enovelada como a nativa.

In [ ]:
#@title <font color="green"> 11.7.6.1 — Fração de contatos nativos
CUTOFF = 0.45   # nm (= 4,5 Å)

# Contatos nativos definidos na estrutura de referência
d_nat, pares = mdtraj.compute_contacts(nativa, contacts='all', scheme='closest-heavy')
mask = d_nat[0] < CUTOFF
pares_nativos = pares[mask]
print("Número de contatos nativos de referência:", pares_nativos.shape[0])

# Mede esses mesmos contatos ao longo da trajetória
d_traj, _ = mdtraj.compute_contacts(traj, contacts=pares_nativos, scheme='closest-heavy')
Q = (d_traj < CUTOFF).mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(tempo_ps, Q, color='purple', lw=1.3)
plt.title('Fração de contatos nativos (Q) ao longo do enovelamento')
plt.xlabel('Tempo (ps)'); plt.ylabel('Q (fração)'); plt.ylim(0, 1)
plt.tight_layout(); plt.show()
print("Q inicial: %.2f  ->  Q final: %.2f" % (Q[0], Q[-1]))

## <font color='magenta'>**11.8 Aplicações em química medicinal (opcional)**</font>

**Flexibilidade do alvo e docking em ensemble.** A conformação cristalográfica é uma entre muitas. Extrair conformações representativas de uma trajetória e ancorar contra todas, no esquema do complexo relaxado (Lin e colaboradores, 2002), recupera parte do encaixe induzido que o docking rígido perde.

**Bolsos crípticos.** Cavidades ausentes na estrutura apo, que se abrem transitoriamente. Aparecem apenas em simulação, e diversos alvos considerados não tratáveis foram reabilitados por essa via.

**Termodinâmica da água no sítio.** Métodos que mapeiam entalpia e entropia de moléculas de água em cada região do sítio, a partir de trajetórias com solvente explícito (Young e colaboradores, 2007), identificam águas cujo deslocamento por um substituinte apolar rende ganho de afinidade. É um racional de desenho originado inteiramente da dinâmica molecular.

**Cinética de ligação.** A afinidade não esgota o problema: o tempo de residência do complexo frequentemente correlaciona melhor com eficácia *in vivo* do que $K_d$ (Copeland, Pompliano e Meek, 2006). Estimar constantes de dissociação exige amostragem de eventos raros, e constitui uma das fronteiras ativas da área.

**Estabilidade de complexos previstos.** Um uso simples e subutilizado consiste em submeter uma pose de docking a alguns nanossegundos de dinâmica e verificar se ela sobrevive. Poses artefatuais frequentemente se desfazem nos primeiros passos. É a aplicação que conecta diretamente algumas seções deste notebook, e um caminho natural para quem for adaptar este material ao próprio alvo.

## <font color='green'>**11.9 Considerações finais sobre a dinâmica**</font>

**O enovelamento é estocástico.** Em 5 ns costumamos observar o colapso da cadeia e o início do enovelamento: queda do raio de giro e do RMSD, subida do Q. Chegar de forma confiável ao estado nativo completo pode exigir trajetórias de dezenas a centenas de nanossegundos, múltiplas réplicas, ou métodos de amostragem acelerada como REMD e metadinâmica. Rode a atividade mais de uma vez e compare: cada execução é diferente, e isso não é defeito, é a natureza do fenômeno.

**Solvente implícito e explícito são escolhas com consequências.** O modelo GB acelera muito o enovelamento e é ideal para demonstração. O solvente explícito, com moléculas de água de verdade, é mais realista e bem mais caro. Se você for simular um complexo proteína-ligante para valer, é explícito que se usa.

**Por que OpenMM, py3Dmol e MDTraj?** É uma pilha inteiramente em Python, aberta, que roda na GPU do próprio Colab sem depender de binários externos, e com visualização estável no navegador. É uma porta de entrada razoável para quem vai trabalhar com estabilidade de proteínas, efeito de mutações ou desenho de fármacos.



## <font color='magenta'>**11.10 Baixando os arquivos gerados (opcional)**</font>

A trajetória e a topologia podem ser abertas depois em VMD, PyMOL, Chimera ou MDAnalysis, no seu próprio computador.

In [ ]:
#@title <font color="magenta"> 11.10.1 — Baixando os arquivos gerados
# (Opcional) Baixar os arquivos gerados para o seu computador
from google.colab import files
files.download('trajetoria.dcd')   # trajetória (abra com VMD, PyMOL, MDAnalysis...)
files.download('topologia.pdb')    # topologia correspondente

## <font color='magenta'>**11.11 Desafio 3**</font>

1. Rode a dinâmica duas vezes com o mesmo `n_steps`, sem mudar nada. As curvas de RMSD e de Q coincidem? Por que não? O que isso implica para quem reporta uma única trajetória num artigo?
2. Reduza `n_steps` para 100.000 e compare o raio de giro final com o da corrida longa. Qual dos dois processos você consegue observar em 200 ps: o colapso da cadeia ou o enovelamento completo?
3. O Trp6 é o núcleo da mini-proteína. Adapte a célula 11.7.4 para calcular a área acessível ao solvente desse resíduo ao longo do tempo (`mdtraj.shrake_rupley`). Ele fica enterrado antes ou depois de o RMSD cair?

# <font color='green'>**12. Encerramento da Parte 02**</font>

## <font color='green'>**12.1 O caminho percorrido**</font>

```
 ┌───────────────┐   ┌───────────────┐   ┌───────────────┐
 │   PARTE 01    │   │  ESTRUTURA 3D │   │  ENCAIXE NO   │
 │  a molécula   │──►│   do ligante  │──►│     ALVO      │──┐
 │virou texto e  │   │               │   │    docking    │  │
 │    número     │   │               │   │               │  │
 └───────────────┘   └───────────────┘   └───────────────┘  │
                                                            │
       ┌────────────────────────────────────────────────────┘
       ▼
 ┌──────────────────────┐   ┌───────────────┐
 │  O TEMPO É IMPORTANTE│   │   PARTE 03    │
 │    dinâmica          │──►│  os elétrons  │
 │   molecular          │   │  voltam à cena│
 └──────────────────────┘   └───────────────┘
```

---

## <font color='green'>**12.2 O que vem na Parte 03**</font>

Estrutura eletrônica com o MOPAC. Até aqui os elétrons ficaram escondidos dentro de parâmetros: campo de força não tem elétron, tem mola, carga fixa e potencial de Lennard-Jones. Na Parte 03 eles voltam à cena.

Métodos semiempíricos usam mecânica quântica, com função de onda e hamiltoniano, mas com as integrais caras computacionalmente substituídas por parâmetros ajustados a dados experimentais. Rodam cem vezes mais rápido que DFT e tratam sistemas de centenas de átomos num Jupyter notebook de forma factível.

Vamos calcular cargas atômicas, orbitais moleculares de fronteira, momento de dipolo e entalpias de biomoléculas.

## <font color='magenta'>**12.3 Avaliação do minicurso**</font>

Se você ainda não respondeu, dois minutos do seu tempo fazem diferença real para a próxima edição. O formulário pergunta o que funcionou, o que ficou confuso e o que você gostaria de ver com mais profundidade.

**[Responder o formulário de avaliação](https://forms.gle/SsPQtyUEqPgEKzat8)**

Rode a célula abaixo para gerar o QR code e responder pelo celular.

In [ ]:
#@title <font color="magenta"> 12.3.1 — QR code do formulário de avaliação
# Cole abaixo o link do seu formulário do Google Forms e rode a célula.

URL_FORMULARIO = "https://forms.gle/SsPQtyUEqPgEKzat8"  #@param {type:"string"}

print("Link do formulário:", URL_FORMULARIO, "\n")

try:
    import qrcode
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qrcode[pil]"])
    import qrcode

import matplotlib.pyplot as plt

img = qrcode.make(URL_FORMULARIO)

fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.imshow(img, cmap="gray")
ax.axis("off")
ax.set_title("Aponte a câmera do celular", fontsize=11, pad=12)
plt.tight_layout()
plt.show()

## <font color='green'>**12.4 Como citar**</font>

Se você usar o gnina num trabalho, cite os autores. Trata-se de software de qualidade disponibilizado livremente, e a citação é a forma de sustentar esse modelo.

1. McNutt AT, Li Y, Meli R, Aggarwal R, Koes DR. **GNINA 1.3: the next increment in molecular docking with deep learning.** *J Cheminform.* 2025.
2. McNutt AT, Francoeur P, Aggarwal R, Masuda T, Meli R, Ragoza M, Sunseri J, Koes DR. **GNINA 1.0: molecular docking with deep learning.** *J Cheminform.* 2021;13:43.
3. Ragoza M, Hochuli J, Idrobo E, Sunseri J, Koes DR. **Protein–ligand scoring with convolutional neural networks.** *J Chem Inf Model.* 2017;57(4):942–957.
4. Sunseri J, Koes DR. **Virtual screening with Gnina 1.0.** *Molecules.* 2021;26(23):7369.
5. O'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. **Open Babel: an open chemical toolbox.** *J Cheminform.* 2011;3:33.
6. Eastman P, et al. **OpenMM 8: molecular dynamics simulation with machine learning potentials.** *J Phys Chem B.* 2024;128(1):109–116.
7. McGibbon RT, et al. **MDTraj: a modern open library for the analysis of molecular dynamics trajectories.** *Biophys J.* 2015;109(8):1528–1532.
8. Neidigh JW, Fesinmeyer RM, Andersen NH. **Designing a 20-residue protein.** *Nat Struct Biol.* 2002;9(6):425–430. (a TRP-cage)

---

**Adaptação:** Prof. Gerd Bruno Rocha — gbr@academico.ufpb.br
Laboratório de Química Quântica Computacional · UFPB
[www.quantum-chem.pro.br](https://www.quantum-chem.pro.br/) · [github.com/RochaGerd/Chemistry_with_Python](https://github.com/RochaGerd/Chemistry_with_Python)

**Material original da Atividade 1:** David Ryan Koes, University of Pittsburgh — *RSC CICAG Open Source Tools for Chemistry Workshops*

*Minicurso "Quimioinformática e Modelagem Molecular — Perspectivas para a Pesquisa em Saúde"*

## <font color='green'>**12.5 Referências**</font>

As referências assinaladas com DOI foram conferidas. As demais trazem dados bibliográficos completos para localização.

**Fundamentos termodinâmicos do docking**

1. Gilson, M. K.; Given, J. A.; Bush, B. L.; McCammon, J. A. The statistical-thermodynamic basis for computation of binding affinities: a critical review. *Biophys. J.* **1997**, *72*, 1047–1069.
2. Gilson, M. K.; Zhou, H.-X. Calculation of protein-ligand binding affinities. *Annu. Rev. Biophys. Biomol. Struct.* **2007**, *36*, 21–42.
3. Koshland, D. E. Application of a theory of enzyme specificity to protein synthesis. *Proc. Natl. Acad. Sci. USA* **1958**, *44*, 98–104.

**Algoritmos de busca e programas de docking**

4. Kuntz, I. D.; Blaney, J. M.; Oatley, S. J.; Langridge, R.; Ferrin, T. E. A geometric approach to macromolecule-ligand interactions. *J. Mol. Biol.* **1982**, *161*, 269–288.
5. Goodford, P. J. A computational procedure for determining energetically favorable binding sites on biologically important macromolecules. *J. Med. Chem.* **1985**, *28*, 849–857.
6. Rarey, M.; Kramer, B.; Lengauer, T.; Klebe, G. A fast flexible docking method using an incremental construction algorithm. *J. Mol. Biol.* **1996**, *261*, 470–489.
7. Jones, G.; Willett, P.; Glen, R. C.; Leach, A. R.; Taylor, R. Development and validation of a genetic algorithm for flexible docking. *J. Mol. Biol.* **1997**, *267*, 727–748.
8. Morris, G. M. *et al.* Automated docking using a Lamarckian genetic algorithm and an empirical binding free energy function. *J. Comput. Chem.* **1998**, *19*, 1639–1662.
9. Trott, O.; Olson, A. J. AutoDock Vina: improving the speed and accuracy of docking with a new scoring function, efficient optimization, and multithreading. *J. Comput. Chem.* **2010**, *31*, 455–461. [DOI: 10.1002/jcc.21334](https://doi.org/10.1002/jcc.21334)
10. Eberhardt, J.; Santos-Martins, D.; Tillack, A. F.; Forli, S. AutoDock Vina 1.2.0: new docking methods, expanded force field, and Python bindings. *J. Chem. Inf. Model.* **2021**, *61*, 3891–3898. [DOI: 10.1021/acs.jcim.1c00203](https://doi.org/10.1021/acs.jcim.1c00203)

**Funções de pontuação**

11. Böhm, H.-J. The development of a simple empirical scoring function to estimate the binding constant for a protein-ligand complex. *J. Comput.-Aided Mol. Des.* **1994**, *8*, 243–256.
12. Muegge, I.; Martin, Y. C. A general and fast scoring function for protein-ligand interactions. *J. Med. Chem.* **1999**, *42*, 791–804.
13. Gohlke, H.; Hendlich, M.; Klebe, G. Knowledge-based scoring function to predict protein-ligand interactions. *J. Mol. Biol.* **2000**, *295*, 337–356.
14. Kitchen, D. B.; Decornez, H.; Furr, J. R.; Bajorath, J. Docking and scoring in virtual screening for drug discovery. *Nat. Rev. Drug Discov.* **2004**, *3*, 935–949.
15. Ballester, P. J.; Mitchell, J. B. O. A machine learning approach to predicting protein-ligand binding affinity. *Bioinformatics* **2010**, *26*, 1169–1175.
16. Ragoza, M.; Hochuli, J.; Idrobo, E.; Sunseri, J.; Koes, D. R. Protein-ligand scoring with convolutional neural networks. *J. Chem. Inf. Model.* **2017**, *57*, 942–957.

**Validação, conjuntos de referência e avaliação crítica**

17. Charifson, P. S.; Corkery, J. J.; Murcko, M. A.; Walters, W. P. Consensus scoring. *J. Med. Chem.* **1999**, *42*, 5100–5109.
18. Warren, G. L. *et al.* A critical assessment of docking programs and scoring functions. *J. Med. Chem.* **2006**, *49*, 5912–5931.
19. Truchon, J.-F.; Bayly, C. I. Evaluating virtual screening methods: good and bad metrics for the early recognition problem. *J. Chem. Inf. Model.* **2007**, *47*, 488–508.
20. Mysinger, M. M.; Carchia, M.; Irwin, J. J.; Shoichet, B. K. Directory of Useful Decoys, Enhanced (DUD-E). *J. Med. Chem.* **2012**, *55*, 6582–6594.
21. Su, M. *et al.* Comparative assessment of scoring functions: the CASF-2016 update. *J. Chem. Inf. Model.* **2019**, *59*, 895–913.
22. Tran-Nguyen, V.-K.; Jacquemard, C.; Rognan, D. LIT-PCBA: an unbiased data set for machine learning and virtual screening. *J. Chem. Inf. Model.* **2020**, *60*, 4263–4273.
23. Buttenschoen, M.; Morris, G. M.; Deane, C. M. PoseBusters: AI-based docking methods fail to generate physically valid poses or generalise to novel sequences. *Chem. Sci.* **2024**, *15*, 3130–3139. [DOI: 10.1039/d3sc04185a](https://doi.org/10.1039/d3sc04185a)
24. Harris, C. *et al.* Benchmarking generated poses: how rational is structure-based drug design with generative models? *arXiv*:2308.07413, **2023**.
25. Tran-Nguyen, V.-K.; Junaid, M.; Simeon, S.; Ballester, P. J. A practical guide to machine-learning scoring for structure-based virtual screening. *Nat. Protoc.* **2023**, *18*, 3460–3511. [DOI: 10.1038/s41596-023-00885-w](https://doi.org/10.1038/s41596-023-00885-w)

**Flexibilidade do receptor, re-ranqueamento e energia livre**

26. Lin, J.-H.; Perryman, A. L.; Schames, J. R.; McCammon, J. A. Computational drug design accommodating receptor flexibility: the relaxed complex scheme. *J. Am. Chem. Soc.* **2002**, *124*, 5632–5633.
27. Sherman, W.; Day, T.; Jacobson, M. P.; Friesner, R. A.; Farid, R. Novel procedure for modeling ligand/receptor induced fit effects. *J. Med. Chem.* **2006**, *49*, 534–553.
28. Genheden, S.; Ryde, U. The MM/PBSA and MM/GBSA methods to estimate ligand-binding affinities. *Expert Opin. Drug Discov.* **2015**, *10*, 449–461.
29. Wang, L. *et al.* Accurate and reliable prediction of relative ligand binding potency in prospective drug discovery. *J. Am. Chem. Soc.* **2015**, *137*, 2695–2703.
30. Schindler, C. E. M. *et al.* Large-scale assessment of binding free energy calculations in active drug discovery projects. *J. Chem. Inf. Model.* **2020**, *60*, 5457–5474. [DOI: 10.1021/acs.jcim.0c00900](https://doi.org/10.1021/acs.jcim.0c00900)
31. Pecina, A.; Fanfrlík, J.; Lepšík, M.; Řezáč, J. SQM2.20: semiempirical quantum-mechanical scoring function yields DFT-quality protein–ligand binding affinity predictions in minutes. *Nat. Commun.* **2024**, *15*, 1127. [DOI: 10.1038/s41467-024-45431-8](https://doi.org/10.1038/s41467-024-45431-8)

**Desenho baseado em estrutura e em fragmentos**

32. Douangamath, A. *et al.* Crystallographic and electrophilic fragment screening of the SARS-CoV-2 main protease. *Nat. Commun.* **2020**, *11*, 5047. [DOI: 10.1038/s41467-020-18709-w](https://doi.org/10.1038/s41467-020-18709-w)
33. Boby, M. L. *et al.* (COVID Moonshot Consortium). Open science discovery of potent noncovalent SARS-CoV-2 main protease inhibitors. *Science* **2023**, *382*, eabo7201.
34. Jumper, J. *et al.* Highly accurate protein structure prediction with AlphaFold. *Nature* **2021**, *596*, 583–589.
35. Abramson, J. *et al.* Accurate structure prediction of biomolecular interactions with AlphaFold 3. *Nature* **2024**, *630*, 493–500.
36. Karelina, M.; Noh, J. J.; Dror, R. O. How accurately can one predict drug binding modes using AlphaFold models? *eLife* **2023**, *12*, RP89386.

**Dinâmica molecular: fundamentos e algoritmos**

37. Alder, B. J.; Wainwright, T. E. Phase transition for a hard sphere system. *J. Chem. Phys.* **1957**, *27*, 1208–1209.
38. McCammon, J. A.; Gelin, B. R.; Karplus, M. Dynamics of folded proteins. *Nature* **1977**, *267*, 585–590. [DOI: 10.1038/267585a0](https://doi.org/10.1038/267585a0)
39. Verlet, L. Computer experiments on classical fluids. I. *Phys. Rev.* **1967**, *159*, 98–103.
40. Swope, W. C.; Andersen, H. C.; Berens, P. H.; Wilson, K. R. A computer simulation method for the calculation of equilibrium constants. *J. Chem. Phys.* **1982**, *76*, 637–649.
41. Ryckaert, J.-P.; Ciccotti, G.; Berendsen, H. J. C. Numerical integration of the cartesian equations of motion of a system with constraints (SHAKE). *J. Comput. Phys.* **1977**, *23*, 327–341.
42. Hess, B.; Bekker, H.; Berendsen, H. J. C.; Fraaije, J. G. E. M. LINCS: a linear constraint solver for molecular simulations. *J. Comput. Chem.* **1997**, *18*, 1463–1472.
43. Miyamoto, S.; Kollman, P. A. SETTLE. *J. Comput. Chem.* **1992**, *13*, 952–962.
44. Tuckerman, M.; Berne, B. J.; Martyna, G. J. Reversible multiple time scale molecular dynamics. *J. Chem. Phys.* **1992**, *97*, 1990–2001.
45. Darden, T.; York, D.; Pedersen, L. Particle mesh Ewald. *J. Chem. Phys.* **1993**, *98*, 10089–10092.

**Termostatos, barostatos e amostragem**

46. Andersen, H. C. Molecular dynamics simulations at constant pressure and/or temperature. *J. Chem. Phys.* **1980**, *72*, 2384–2393.
47. Berendsen, H. J. C. *et al.* Molecular dynamics with coupling to an external bath. *J. Chem. Phys.* **1984**, *81*, 3684–3690.
48. Nosé, S. A unified formulation of the constant temperature molecular dynamics methods. *J. Chem. Phys.* **1984**, *81*, 511–519.
49. Hoover, W. G. Canonical dynamics: equilibrium phase-space distributions. *Phys. Rev. A* **1985**, *31*, 1695–1697.
50. Parrinello, M.; Rahman, A. Polymorphic transitions in single crystals. *J. Appl. Phys.* **1981**, *52*, 7182–7190.
51. Harvey, S. C.; Tan, R. K.-Z.; Cheatham, T. E. The flying ice cube. *J. Comput. Chem.* **1998**, *19*, 726–740.
52. Bussi, G.; Donadio, D.; Parrinello, M. Canonical sampling through velocity rescaling. *J. Chem. Phys.* **2007**, *126*, 014101.
53. Leimkuhler, B.; Matthews, C. Rational construction of stochastic numerical methods for molecular sampling. *Appl. Math. Res. eXpress* **2013**, *2013*, 34–56. [DOI: 10.1093/amrx/abs010](https://doi.org/10.1093/amrx/abs010)
54. Torrie, G. M.; Valleau, J. P. Nonphysical sampling distributions in Monte Carlo free-energy estimation: umbrella sampling. *J. Comput. Phys.* **1977**, *23*, 187–199.
55. Sugita, Y.; Okamoto, Y. Replica-exchange molecular dynamics method for protein folding. *Chem. Phys. Lett.* **1999**, *314*, 141–151.
56. Laio, A.; Parrinello, M. Escaping free-energy minima. *Proc. Natl. Acad. Sci. USA* **2002**, *99*, 12562–12566.
57. Barducci, A.; Bussi, G.; Parrinello, M. Well-tempered metadynamics. *Phys. Rev. Lett.* **2008**, *100*, 020603.
58. Shirts, M. R.; Chodera, J. D. Statistically optimal analysis of samples from multiple equilibrium states (MBAR). *J. Chem. Phys.* **2008**, *129*, 124105.

**Análise, estatística e aplicações**

59. Amadei, A.; Linssen, A. B. M.; Berendsen, H. J. C. Essential dynamics of proteins. *Proteins* **1993**, *17*, 412–425.
60. Flyvbjerg, H.; Petersen, H. G. Error estimates on averages of correlated data. *J. Chem. Phys.* **1989**, *91*, 461–466.
61. Grossfield, A.; Zuckerman, D. M. Quantifying uncertainty and sampling quality in biomolecular simulations. *Annu. Rep. Comput. Chem.* **2009**, *5*, 23–48.
62. Young, T.; Abel, R.; Kim, B.; Berne, B. J.; Friesner, R. A. Motifs for molecular recognition exploiting hydrophobic enclosure. *Proc. Natl. Acad. Sci. USA* **2007**, *104*, 808–813.
63. Copeland, R. A.; Pompliano, D. L.; Meek, T. D. Drug-target residence time and its implications for lead optimization. *Nat. Rev. Drug Discov.* **2006**, *5*, 730–739.
64. Shaw, D. E. *et al.* Atomic-level characterization of the structural dynamics of proteins. *Science* **2010**, *330*, 341–346.
65. Lindorff-Larsen, K.; Piana, S.; Dror, R. O.; Shaw, D. E. How fast-folding proteins fold. *Science* **2011**, *334*, 517–520.
66. Neidigh, J. W.; Fesinmeyer, R. M.; Andersen, N. H. Designing a 20-residue protein. *Nat. Struct. Biol.* **2002**, *9*, 425–430.

**Programas de código aberto usados neste notebook**

67. McNutt, A. T. *et al.* GNINA 1.0: molecular docking with deep learning. *J. Cheminform.* **2021**, *13*, 43.
68. Sunseri, J.; Koes, D. R. Virtual screening with Gnina 1.0. *Molecules* **2021**, *26*, 7369.
69. O'Boyle, N. M. *et al.* Open Babel: an open chemical toolbox. *J. Cheminform.* **2011**, *3*, 33.
70. Eastman, P. *et al.* OpenMM 8: molecular dynamics simulation with machine learning potentials. *J. Phys. Chem. B* **2024**, *128*, 109–116. [DOI: 10.1021/acs.jpcb.3c06662](https://doi.org/10.1021/acs.jpcb.3c06662)
71. McGibbon, R. T. *et al.* MDTraj: a modern open library for the analysis of molecular dynamics trajectories. *Biophys. J.* **2015**, *109*, 1528–1532.

**Trabalhos do grupo sobre re-ranqueamento quântico de poses**

72. Rocha, G. B. e colaboradores. *ACS Omega* **2020**, *5*. [DOI: 10.1021/acsomega.0c02588](https://doi.org/10.1021/acsomega.0c02588)
73. Rocha, G. B. e colaboradores. *J. Biomol. Struct. Dyn.* **2021**. [DOI: 10.1080/07391102.2021.1878058](https://doi.org/10.1080/07391102.2021.1878058)

**Livros**

74. Leach, A. R. *Molecular Modelling: Principles and Applications*, 2ª ed. Prentice Hall, 2001.
75. Klebe, G. *Drug Design: Methodology, Concepts, and Mode-of-Action*. Springer, 2013.
76. Frenkel, D.; Smit, B. *Understanding Molecular Simulation*, 2ª ed. Academic Press, 2002.
77. Allen, M. P.; Tildesley, D. J. *Computer Simulation of Liquids*, 2ª ed. Oxford University Press, 2017.
78. Tuckerman, M. E. *Statistical Mechanics: Theory and Molecular Simulation*. Oxford University Press, 2010.

## <font color='magenta'>**12.6 Para continuar (opcional)**</font>

**Dinâmica molecular em solvente explícito.** É o passo natural depois desta atividade, e o material do Pablo Arantes resolve o problema de infraestrutura para você:

- [Making it rain](https://github.com/pablo-arantes/Making-it-rain) — notebooks de dinâmica molecular no Colab
- [Versão com AmberTools](https://colab.research.google.com/github/pablo-arantes/Making-it-rain/blob/main/Amber.ipynb)
- [Notebooks científicos do Colab](https://colab.google/notebooks/#science)

**Análise de interações proteína-ligante.** O ProLIF já está instalado neste notebook e não chegamos a usá-lo. Ele identifica ligações de hidrogênio, contatos hidrofóbicos, empilhamento π e pontes salinas, frame a frame ao longo de uma trajetória. O interessante não é a foto de um frame, e sim a **persistência** de cada interação ao longo do tempo:

- [Documentação e tutoriais do ProLIF](https://prolif.readthedocs.io/)

**Predição de estrutura.** Quando não existe estrutura experimental do seu alvo:

- [AlphaFold DB](https://alphafold.ebi.ac.uk/) — modelos preditos para praticamente todo proteoma conhecido
- [ColabFold](https://github.com/sokrypton/ColabFold) — AlphaFold rodando no Colab

Uma advertência sobre modelos preditos em docking: a cadeia principal costuma vir boa, as cadeias laterais do sítio nem sempre. Vale um cross-docking de validação antes de confiar.

---